# E8-N v3 — The Budget Cure (Phase 10, UI flight)

**Pre-registration: `docs/E8N3_PROTOCOL.md` (session 134, commit 26f17b0) —
locks at first full flight.**

ONE training change from the locked v2: **per-strand plateau** (the minted
lesson, donor code that flew E8-F v2 and E8-O2) at **EPOCH_CAP 12** — the
design check projects the competence strand reaches base's passing loss
level ~epoch 11. Everything else is v2-byte-identical: same 600-example
joint curriculum, same LoRA, same eval instruments. **REAL condition only**
(base's answer is locked), so the full flight is ONE run.

Primaries (Holm-2): **P-V1** catch ≥ 9/12 + binomial improvement over this
flight's own pre (P2 on budget — the rung) · **P-V2** battery tracking
survives (retention). **S0**: instillation-alone tracking, third
prospective replication (band [.1, .3]). **S10′**: forced-choice
comprehension vs the PINNED v2 per-row baseline — the FC-interference
backlog item measured by paired test (readings pre-stated). **S-ABS**:
absolute-calibration rows re-measured (the arm-split finding's prospective
replication; predictions registered).

**This notebook is SELF-CONTAINED** (battery, pools, locked rows, catch,
competence, lexicon, FC baseline all embedded). Drive I/O via `drive.mount`
only: the E4 adapter, the dictionary pack, the shipped E8-R bundle dirs
(stability gate), and shipping to `MyDrive/semcore/e8n3/`.

**How to run (Joe):** Runtime → Change runtime type → **T4 GPU** → Run all.
First run uses `SMOKE = True` (~10–14 min) and ends in a green or red
banner — mechanics only.

**Full flight = ONE run:** flip `SMOKE = False` → **Runtime → Restart
runtime** → Run all (~90–120 min worst case; per-strand plateau may stop
training earlier). If the VM dies mid-run, paste the banner's
`RESUME_STAMP = '...'` into the config cell → Restart → Run all; the
finished condition reloads from Drive in seconds. Restarts between smoke
and full are MANDATORY; the setup cell refuses dirty kernels.

In [ ]:
# ── Config + setup: GPU, installs, Drive mount, pack, adapter, E8-R bundles ──
NB_BUILD = 'v1 (2026-08-25)'
print('E8-N v3 notebook build:', NB_BUILD)

SMOKE = True                   # first run: smoke (~12-16 min). Then False.
RESUME_STAMP = ''              # paste the banner's stamp between full runs
ONE_CONDITION_PER_RUN = True   # v3: single condition, one run
CONDITIONS = ('real',)         # v3 is REAL-only (base locked)

import subprocess, sys, os, json, re, math, time, shutil, gc, ctypes
from pathlib import Path
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['MALLOC_ARENA_MAX'] = '2'

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')

print('Installing packages...')
subprocess.run([sys.executable,'-m','pip','uninstall','-q','-y','torchao'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
    'transformers>=4.44','peft>=0.11','accelerate','scipy',
    'sentence-transformers>=3.0'], check=True)

import torch
assert torch.cuda.is_available(), 'No GPU — Runtime > Change runtime type > T4 GPU.'
DEV = 'cuda'

def _mem_avail_gb():
    try:
        kb = int(next(l for l in open('/proc/meminfo')
                      if l.startswith('MemAvailable')).split()[1])
        return kb / 1e6
    except Exception:
        return float('nan')

def free_ram():
    gc.collect()
    torch.cuda.empty_cache()
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass

def ram_report():
    g = torch.cuda.mem_get_info()
    return (f'sys avail {_mem_avail_gb():.1f}GB | '
            f'GPU free {g[0]/1e9:.1f}/{g[1]/1e9:.1f}GB')

_leftover = torch.cuda.memory_allocated()
assert _leftover < 5e8, (
    f'GPU already holds {_leftover/1e9:.1f}GB from a previous run in this '
    'kernel — this flight needs a fresh one. Runtime > Restart runtime, '
    'then Run all.')
_avail = _mem_avail_gb()
_floor = 6.5
assert not (_avail < _floor), (
    f'Only {_avail:.1f}GB system RAM available (need {_floor}). Runtime > '
    'Restart runtime; if it trips again, Runtime > Disconnect and delete '
    'runtime for a fresh VM, then Run all.')
print('RAM at start:', ram_report())

from google.colab import drive
drive.mount('/content/drive')
SEM = Path('/content/drive/MyDrive/semcore')
assert SEM.exists(), 'MyDrive/semcore not found — mounted the right Google account?'

def ship(src, dest_rel):
    dest = SEM / dest_rel
    dest.mkdir(parents=True, exist_ok=True)
    src = Path(src)
    files = sorted(p for p in src.iterdir() if p.is_file()) if src.is_dir() else [src]
    for p in files:
        shutil.copy2(p, dest / p.name)

PACK = Path('/content/e4_dictionary_pack.json')
if not PACK.exists():
    shutil.copy2(SEM / 'e4/e4_dictionary_pack.json', PACK)
pack = json.load(open(PACK))
print('pack:', pack['name'], '| concepts', pack['n_concepts'])
VEC = {c['name']: c['vec'] for c in pack['concepts']}
DESC = {c['name']: c['desc'] for c in pack['concepts']}

# Shipped E8-R stimulus (dirs-stability gate rides E8-R2's measured residual)
E8R_SRC = SEM / 'e8r/inflight_20260822_2329'   # flight of record — never change
SHIPPED = {}
for _c in ('real',):
    _fp = E8R_SRC / f'condition_{_c}.json'
    assert _fp.exists(), (f'missing E8-R bundle {_fp} — the dirs-stability '
                          'gate needs the flight-of-record stimulus')
    _b = json.load(open(_fp))
    assert _b.get('dirs') and _b.get('mu'), f'{_c}: bundle lacks dirs/mu'
    SHIPPED[_c] = {'dirs': _b['dirs'], 'mu': _b['mu']}
    print(f'E8-R shipped stimulus [{_c}]: layers', sorted(_b['dirs']))

ADAPTERS = {}
cand = sorted(d.name for d in (SEM / 'e4').iterdir()
              if d.is_dir() and d.name.startswith('real_full_'))
assert cand, 'no real_full_* dir under semcore/e4'
src = SEM / 'e4' / cand[-1] / 'adapter_real'
dst = Path('/content/adapter_real')
shutil.copytree(src, dst, dirs_exist_ok=True)
assert (dst / 'adapter_config.json').exists(), 'adapter_real incomplete'
ADAPTERS['real'] = str(dst)
print('adapter real:', cand[-1])

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
STAMP = time.strftime('%Y%m%d_%H%M')
MODE = 'smoke' if SMOKE else 'full'
OUT = Path(f'/content/out_{MODE}_{STAMP}'); OUT.mkdir(parents=True, exist_ok=True)
INFLIGHT = f'e8n3/inflight_{RESUME_STAMP or STAMP}'
if RESUME_STAMP:
    _rd = SEM / INFLIGHT
    assert _rd.exists(), (
        f'RESUME_STAMP={RESUME_STAMP!r} but {_rd} does not exist on Drive — '
        'check the stamp string (copy it exactly; no spaces). A silent '
        'fallback here would re-fly finished conditions.')
    _have = sorted(p.name for p in _rd.glob('condition_*.json'))
    print('resume dir found; bundles present:', _have or 'NONE')
print('MODE:', MODE.upper(), '| stamp', STAMP,
      ('| RESUMING ' + RESUME_STAMP) if RESUME_STAMP else '')


In [ ]:
# ── E8N pure logic: labels, scoring, stats, validators (locally tested verbatim) ──
import re, math
import numpy as np

E8N_SEED = 20260824          # E8-R took 20260823; fresh stream for this rung
ARMS = ['uncertainty', 'familiarity', 'tension', 'saturation']
TAU = 0.25                   # convergence target: smoothed train loss at epoch end
EPOCHS_CAP = 8               # hard cap (~2x E8-R's opt-step budget)
CATCH_TOL = 2                # catch trial passes at |report - known| <= 2
CATCH_PASS_MIN = 9           # P-E8N-2 clause (b): >= 9/12
TOOK_MIN_FRAC = 0.60         # train-took gate: within +/-1 on >= 60%
PPL_GATE_PCT = 5.0
N_PERM = 2000
N_BOOT = 10000
MIN_POOLED_N = 8

INT_RE = re.compile(r'\b(10|[0-9])\b')          # E5 verbatim

def unflip(val, flipped):                        # E5 verbatim
    return None if val is None else (10 - val if flipped else val)

def canon(s):                                    # E5 verbatim
    s = re.sub(r'[^a-z0-9 ]', '', s.lower())
    s = re.sub(r'^(the|a|an) ', '', s.strip())
    return ' '.join(s.split()[:8])

def answer_slice(prompt_len, total_len):
    """Hidden-state index range whose logits predict the answer tokens:
    position p predicts token p+1, so predicting ids[prompt_len:total_len]
    takes hidden[prompt_len-1 : total_len-1]. (v2 memory law, smoke-1 OOM:
    training must never materialize full-sequence logits — slice the head.)"""
    assert 0 < prompt_len < total_len, (prompt_len, total_len)
    return prompt_len - 1, total_len - 1

def jdump(obj, path, indent=1):
    """json.dump with numpy-scalar safety (int64/float64/ndarray -> native)."""
    import json as _json
    class _NpEnc(_json.JSONEncoder):
        def default(self, o):
            if isinstance(o, np.integer):
                return int(o)
            if isinstance(o, np.floating):
                return float(o)
            if isinstance(o, np.ndarray):
                return o.tolist()
            return super().default(o)
    with open(path, 'w') as f:
        _json.dump(obj, f, indent=indent, cls=_NpEnc)

# ── firewall: training pools must be disjoint from the locked battery ────────
def norm_text(s):
    return ' '.join(re.sub(r'[^a-z0-9 ]', ' ', s.lower()).split())

def _battery_texts(bat_arms):
    out = []
    for it in bat_arms['uncertainty']['items']:
        out.append(('uncertainty', it['id'], it['text']))
    for it in bat_arms['familiarity']['items']:
        out.append(('familiarity', it['id'], it['text']))
    for it in bat_arms['tension']['items']:
        out.append(('tension', it['id'], it['text']))
    for it in bat_arms['saturation']['items']:
        out.append(('saturation', it['id'], it['needle']))
        out.append(('saturation', it['id'] + 'q', it['question']))
    return out

def _pool_texts(pools):
    out = []
    for arm in ('uncertainty', 'familiarity', 'tension'):
        for it in pools[arm]:
            out.append((arm, it['id'], it['text']))
    for it in pools['saturation']:
        out.append(('saturation', it['id'], it['needle']))
        out.append(('saturation', it['id'] + 'q', it['question']))
    return out

def _shingles(nt, k=8):
    ws = nt.split()
    if len(ws) >= k:
        return {' '.join(ws[i:i + k]) for i in range(len(ws) - k + 1)}
    return {nt} if ws else set()

def validate_disjoint(pools, bat_arms, k=8):
    """Firewall: a training text violates if it (a) equals a battery text
    (normalized), (b) contains / is contained in one (>=20 normalized chars),
    or (c) shares any k-word shingle with one (fragment overlap, both
    directions). Flight refuses if any violation."""
    viols = []
    bats = [(f'{b}/{j}', norm_text(t)) for b, j, t in _battery_texts(bat_arms)]
    bat_sh = {}
    for tag, nu in bats:
        for sh in _shingles(nu, k):
            bat_sh.setdefault(sh, tag)
    for a, i, t in _pool_texts(pools):
        nt = norm_text(t)
        hits = set()
        for tag, nu in bats:
            if nt == nu or (len(nt) >= 20 and nt in nu) or (len(nu) >= 20 and nu in nt):
                hits.add(tag)
        hits |= {bat_sh[sh] for sh in _shingles(nt, k) if sh in bat_sh}
        viols.extend({'pool': f'{a}/{i}', 'battery': tag} for tag in sorted(hits))
    return viols

# ── referent orientation (higher oriented value => higher straight report) ───
def orient_referent(arm, row):
    if arm == 'uncertainty':
        return float(row['entropy'])
    if arm == 'familiarity':
        return -float(row['nll'])
    if arm == 'tension':
        return float(row['divergence'])
    if arm == 'saturation':
        return float(row['fill_fraction'])
    raise KeyError(arm)

# ── rank machinery (tie-averaged, scipy-free) ────────────────────────────────
def rankdata_avg(vals):
    a = np.asarray(vals, float)
    order = np.argsort(a, kind='mergesort')
    ranks = np.empty(len(a), float)
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and a[order[j + 1]] == a[order[i]]:
            j += 1
        ranks[order[i:j + 1]] = (i + j) / 2.0 + 1.0
        i = j + 1
    return ranks

def rank01(vals):
    n = len(vals)
    if n == 1:
        return np.array([0.5])
    return (rankdata_avg(vals) - 1.0) / (n - 1.0)

# ── pooled tracking statistic ────────────────────────────────────────────────
def pooled_rho(arm_rows):
    """arm_rows: {arm: [{'report': int, 'ref': float, ...}, ...]} (reports
    already unflipped; None-report rows excluded upstream). Within-arm
    normalized average-tie ranks of report and referent, pooled, Pearson."""
    xs, ys, per_arm_n = [], [], {}
    for arm, rows in arm_rows.items():
        if not rows:
            per_arm_n[arm] = 0
            continue
        per_arm_n[arm] = len(rows)
        xs.append(rank01([r['report'] for r in rows]))
        ys.append(rank01([r['ref'] for r in rows]))
    if not xs:
        return {'rho': None, 'n': 0, 'per_arm_n': per_arm_n, 'degenerate': True}
    x, y = np.concatenate(xs), np.concatenate(ys)
    n = len(x)
    if n < MIN_POOLED_N or np.std(x) == 0 or np.std(y) == 0:
        return {'rho': None, 'n': n, 'per_arm_n': per_arm_n, 'degenerate': True}
    r = float(np.corrcoef(x, y)[0, 1])
    return {'rho': round(r, 4), 'n': n, 'per_arm_n': per_arm_n, 'degenerate': False}

def perm_p_pooled(arm_rows, n_perm=None, seed=E8N_SEED):
    if n_perm is None:
        n_perm = N_PERM
    """One-sided permutation p for pooled_rho > 0: shuffle the report column
    WITHIN each arm (referents fixed). Degenerate observed => p = 1.0."""
    obs = pooled_rho(arm_rows)
    if obs['rho'] is None:
        return {'rho': None, 'p': 1.0, 'n': obs['n'], 'degenerate': True}
    rng = np.random.default_rng(seed)
    cnt = 0
    for _ in range(n_perm):
        sh = {}
        for arm, rows in arm_rows.items():
            if not rows:
                sh[arm] = rows
                continue
            reps = [r['report'] for r in rows]
            rng.shuffle(reps)
            sh[arm] = [{'report': rep, 'ref': r['ref']} for rep, r in zip(reps, rows)]
        rp = pooled_rho(sh)['rho']
        if rp is not None and rp >= obs['rho']:
            cnt += 1
    return {'rho': obs['rho'], 'p': round((1 + cnt) / (1 + n_perm), 5),
            'n': obs['n'], 'per_arm_n': obs['per_arm_n'], 'degenerate': False}

def boot_rho_ci(arm_rows, n_boot=None, seed=E8N_SEED):
    if n_boot is None:
        n_boot = N_BOOT
    """Stratified (within-arm) item bootstrap CI for pooled_rho."""
    obs = pooled_rho(arm_rows)
    if obs['rho'] is None:
        return {'rho': None, 'ci95': [None, None], 'n': obs['n']}
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        res = {}
        for arm, rows in arm_rows.items():
            if not rows:
                res[arm] = rows
                continue
            idx = rng.integers(0, len(rows), len(rows))
            res[arm] = [rows[i] for i in idx]
        rb = pooled_rho(res)['rho']
        if rb is not None:
            boots.append(rb)
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (None, None))
    return {'rho': obs['rho'], 'ci95': [round(float(lo), 4), round(float(hi), 4)]
            if boots else [None, None], 'n': obs['n']}

def paired_boot_delta_rho(A, B, n_boot=None, seed=E8N_SEED):
    if n_boot is None:
        n_boot = N_BOOT
    """Bootstrap CI for pooled_rho(A) - pooled_rho(B), items paired by id
    within arm (same resample drives both sides)."""
    pairs = {}
    for arm in ARMS:
        ax = {r['id']: r for r in A.get(arm, []) if 'id' in r}
        bx = {r['id']: r for r in B.get(arm, []) if 'id' in r}
        ids = sorted(set(ax) & set(bx))
        if ids:
            pairs[arm] = [(ax[i], bx[i]) for i in ids]
    if not pairs:
        return {'delta': None, 'ci95': [None, None], 'n': 0}
    oa = pooled_rho({a: [p[0] for p in v] for a, v in pairs.items()})['rho']
    ob = pooled_rho({a: [p[1] for p in v] for a, v in pairs.items()})['rho']
    if oa is None or ob is None:
        return {'delta': None, 'ci95': [None, None],
                'n': sum(len(v) for v in pairs.values())}
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        ra, rb = {}, {}
        for arm, v in pairs.items():
            idx = rng.integers(0, len(v), len(v))
            ra[arm] = [v[i][0] for i in idx]
            rb[arm] = [v[i][1] for i in idx]
        da, db = pooled_rho(ra)['rho'], pooled_rho(rb)['rho']
        if da is not None and db is not None:
            boots.append(da - db)
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (None, None))
    return {'delta': round(oa - ob, 4), 'rho_a': oa, 'rho_b': ob,
            'ci95': [round(float(lo), 4), round(float(hi), 4)] if boots else [None, None],
            'n': sum(len(v) for v in pairs.values())}

def split_polarity(arm_rows):
    out = {}
    for flag, name in ((False, 'straight'), (True, 'flipped')):
        out[name] = {arm: [r for r in rows if r.get('flipped') == flag]
                     for arm, rows in arm_rows.items()}
    return out

# ── battery rows -> scoring structures ───────────────────────────────────────
def battery_rows_to_scoring(rows_by_arm):
    """Runner row dicts -> {arm: [{'id','report','ref','flipped'}]} (named rows
    only) + parse-fail counts + per-arm report variance."""
    scoring, meta = {}, {}
    for arm in ARMS:
        rows = rows_by_arm.get(arm) or []
        named = []
        for r in rows:
            if r.get('report') is None:
                continue
            named.append({'id': r['id'] if arm != 'saturation'
                          else f"{r['id']}@{r['target_frac']}",
                          'report': int(r['report']),
                          'ref': orient_referent(arm, r),
                          'flipped': bool(r['flipped'])})
        scoring[arm] = named
        reps = [r['report'] for r in named]
        meta[arm] = {'n': len(rows), 'named': len(named),
                     'parse_fail': len(rows) - len(named),
                     'report_variance': round(float(np.var(reps)), 3) if reps else None}
    return scoring, meta

def per_arm_rho(scoring):
    out = {}
    for arm in ARMS:
        rows = scoring.get(arm) or []
        out[arm] = pooled_rho({arm: rows})
    return out

def catch_score(rows):
    named = [r for r in rows if r.get('report') is not None]
    passed = sum(1 for r in named if abs(r['report'] - r['known']) <= CATCH_TOL)
    return {'n': len(rows), 'named': len(named), 'passed': passed,
            'pass': passed >= CATCH_PASS_MIN}

# ── training-set construction (labels from measured referents) ───────────────
def s_stimuli(pools, fill_fractions):
    return [{'sid': f"{it['id']}@{f}", 'needle_id': it['id'], 'frac': f}
            for it in pools['saturation'] for f in fill_fractions]

def quantile_labels(oriented_vals):
    n = len(oriented_vals)
    if n == 1:
        return [5]
    q = rank01(oriented_vals)
    return [int(round(10 * v)) for v in q]

def build_training_examples(pools, pool_refs, fill_fractions):
    """pool_refs: {'uncertainty': {sid: {'entropy':..}}, 'familiarity':
    {sid: {'nll':..}}, 'tension': {sid: {'divergence':..}}, 'saturation':
    {sid: {'fill_fraction':..}}}. Emits 2 examples per stimulus (straight +
    flipped), labels = within-arm quantiles of the oriented referent."""
    examples, eid = [], 0
    for arm in ARMS:
        if arm == 'saturation':
            stims = s_stimuli(pools, fill_fractions)
            sids = [s['sid'] for s in stims]
        else:
            sids = [it['id'] for it in pools[arm]]
        missing = [s for s in sids if s not in pool_refs[arm]]
        assert not missing, f'{arm}: unmeasured stimuli {missing[:4]}'
        oriented = [orient_referent(arm, pool_refs[arm][s]) for s in sids]
        labels = quantile_labels(oriented)
        for s, lab in zip(sids, labels):
            for flipped in (False, True):
                examples.append({'eid': eid, 'arm': arm, 'sid': s,
                                 'flipped': flipped,
                                 'label': (10 - lab) if flipped else lab})
                eid += 1
    return examples

def took_subset(examples, n_per_arm=6, seed=E8N_SEED + 3):
    rng = np.random.default_rng(seed)
    out = []
    for arm in ARMS:
        straight = [e for e in examples if e['arm'] == arm and not e['flipped']]
        k = min(n_per_arm, len(straight))
        idx = rng.choice(len(straight), size=k, replace=False)
        out.extend(straight[int(i)] for i in sorted(idx))
    return out

def paraphrase_draw(bat_arms, fill_fractions, n_per_arm=3, seed=E8N_SEED + 5):
    """Straight-assigned battery items (index parity: even = straight), drawn
    per arm; S uses the largest fill."""
    rng = np.random.default_rng(seed)
    out = {}
    for arm in ('uncertainty', 'familiarity', 'tension'):
        straight = [it for i, it in enumerate(bat_arms[arm]['items']) if i % 2 == 0]
        k = min(n_per_arm, len(straight))
        idx = rng.choice(len(straight), size=k, replace=False)
        out[arm] = [straight[int(i)]['id'] for i in sorted(idx)]
    s_items = bat_arms['saturation']['items']
    k = min(n_per_arm, len(s_items))
    idx = rng.choice(len(s_items), size=k, replace=False)
    out['saturation'] = [(s_items[int(i)]['id'], max(fill_fractions))
                         for i in sorted(idx)]
    return out

# ── smoke subsetting ─────────────────────────────────────────────────────────
BATTERY_SMOKE = {  # E5 cell-2 smoke ids, verbatim
    'uncertainty': {'U01', 'U08', 'U17', 'U23', 'U33', 'U42'},
    'familiarity': {'F01', 'F06', 'F11', 'F16', 'F21', 'F26', 'F31', 'F36'},
    'tension_bases': {1, 7},
    'saturation': {'S01', 'S06'},
}
POOL_SMOKE = {
    'uncertainty': {'NU01', 'NU02', 'NU17', 'NU18', 'NU33', 'NU34'},
    'familiarity': {'NF01', 'NF06', 'NF11', 'NF16', 'NF21', 'NF26', 'NF31', 'NF36'},
    'tension_bases': {1, 7},
    'saturation': {'NS01', 'NS02'},
}

def smoke_pools(pools):
    return {
        'uncertainty': [it for it in pools['uncertainty'] if it['id'] in POOL_SMOKE['uncertainty']],
        'familiarity': [it for it in pools['familiarity'] if it['id'] in POOL_SMOKE['familiarity']],
        'tension': [it for it in pools['tension'] if it['base'] in POOL_SMOKE['tension_bases']],
        'saturation': [it for it in pools['saturation'] if it['id'] in POOL_SMOKE['saturation']],
    }

def holm(pvals):
    """{name: p} -> {name: (p, reject_at_.05)} Holm step-down (E8-R verbatim)."""
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    m = len(items)
    out, stopped = {}, False
    for i, (name, p) in enumerate(items):
        rej = (not stopped) and (p <= 0.05 / (m - i))
        if not rej:
            stopped = True
        out[name] = (p, rej)
    return out


# ═══ naming + forced-choice machinery: e8r2_logic.py VERBATIM (carries e8r_logic) ═══
# ── E8R pure logic: plans, training set, parser, scoring (locally tested verbatim) ──
import numpy as np

E7Q_SEED = 20260822          # the locked eval instrument's seed — never change
E8R_SEED = 20260823          # all E8-R-new randomness (training set, shams, draws)
CHOICE_SET = ["UNCERTAINTY","CONFIDENCE","TENSION","RESOLUTION","RETRIEVAL",
              "CONSTRUCTION","SATURATION","FAMILIARITY","NOVELTY","CAPTURE",
              "DIVERGENCE","CONFABULATION","CALIBRATION"]
HELD_OUT = ["DIVERGENCE","NOVELTY","RETRIEVAL","TENSION"]   # pinned draw, seed 20260823
TRAINED = [c for c in CHOICE_SET if c not in HELD_OUT]
LAYERS = [14, 20]            # hidden_states indexing (E4/atlas convention)
ALPHAS = [0.25, 0.5, 1.0]
N_SHAMS = 12                 # in the locked E7-Q plan
TRAIN_LAYER = 14
TRAIN_ALPHAS = [0.5, 1.0]
N_TRAIN_ORDERS = 8           # menu orders per (concept, alpha)
N_TRAIN_SHAMS = 72
N_PRE_SHAMS = 12
N_SUPP_SHAMS = 12
N_TRAINTOOK = 18
FA_MAX_CLAIMS = 18           # P-E8R-2 clause (a): sham claims must be <= 18/24

def jdump(obj, path, indent=1):
    """json.dump with numpy-scalar safety (int64/float64/ndarray -> native)."""
    import json as _json
    class _NpEnc(_json.JSONEncoder):
        def default(self, o):
            if isinstance(o, np.integer):
                return int(o)
            if isinstance(o, np.floating):
                return float(o)
            if isinstance(o, np.ndarray):
                return o.tolist()
            return super().default(o)
    with open(path, 'w') as f:
        _json.dump(obj, f, indent=indent, cls=_NpEnc)

def heldout_draw():
    """Reproduces the pinned held-out draw from seed + pre-stated constraints
    (protocol: attractors excluded from eligibility; at most one pole per
    remaining v0 complement pair)."""
    attractors = {"UNCERTAINTY","CONFIDENCE","CALIBRATION","RESOLUTION"}
    pairs = [("RETRIEVAL","CONSTRUCTION"),("FAMILIARITY","NOVELTY")]
    elig = [c for c in CHOICE_SET if c not in attractors]
    rng = np.random.default_rng(E8R_SEED)
    while True:
        draw = sorted(rng.choice(elig, size=4, replace=False).tolist())
        if all(not (a in draw and b in draw) for a, b in pairs):
            return draw

def build_plan(smoke=False):
    """THE LOCKED E7-Q EVAL PLAN — verbatim rng stream (full mode must be
    bit-identical to e7q_logic.build_plan(False)). Smoke mode deviates only
    in its concept list (one trained + one held-out, to exercise both paths)."""
    rng = np.random.default_rng(E7Q_SEED)
    concepts = CHOICE_SET
    layers, alphas, n_shams = LAYERS, ALPHAS, N_SHAMS
    if smoke:
        concepts = ["UNCERTAINTY", "TENSION"]
        layers, alphas, n_shams = [14], [0.5], 3
    plan = []
    tid = 0
    for c in concepts:
        for L in layers:
            for a in alphas:
                order = [int(i) for i in rng.permutation(len(CHOICE_SET))]
                plan.append({"tid": tid, "kind": "inject", "concept": c,
                             "layer": L, "alpha": a, "order": order})
                tid += 1
    for _ in range(n_shams):
        order = [int(i) for i in rng.permutation(len(CHOICE_SET))]
        plan.append({"tid": tid, "kind": "sham", "concept": None,
                     "layer": None, "alpha": 0.0, "order": order})
        tid += 1
    return plan

def build_train_set(smoke=False):
    """Readout training curriculum: TRAINED concepts x TRAIN_ALPHAS x L14 x
    N_TRAIN_ORDERS injected + N_TRAIN_SHAMS shams (target NONE). Seed E8R —
    orders disjoint from the eval plan's by seed separation."""
    rng = np.random.default_rng(E8R_SEED + 1)
    concepts, alphas = TRAINED, TRAIN_ALPHAS
    n_orders, n_shams = N_TRAIN_ORDERS, N_TRAIN_SHAMS
    if smoke:
        concepts, n_orders, n_shams = ["UNCERTAINTY", "CAPTURE"], 2, 4
    ex, eid = [], 0
    for c in concepts:
        for a in alphas:
            for _ in range(n_orders):
                order = [int(i) for i in rng.permutation(len(CHOICE_SET))]
                ex.append({"eid": eid, "kind": "inject", "concept": c,
                           "layer": TRAIN_LAYER, "alpha": a, "order": order,
                           "target": c})
                eid += 1
    for _ in range(n_shams):
        order = [int(i) for i in rng.permutation(len(CHOICE_SET))]
        ex.append({"eid": eid, "kind": "sham", "concept": None, "layer": None,
                   "alpha": 0.0, "order": order, "target": "NONE"})
        eid += 1
    return ex

def build_shams(n, seed, tid_base):
    """Extra sham trials (pre-eval anchor / supplementary), fresh orders."""
    rng = np.random.default_rng(seed)
    out = []
    for i in range(n):
        order = [int(j) for j in rng.permutation(len(CHOICE_SET))]
        out.append({"tid": tid_base + i, "kind": "sham", "concept": None,
                    "layer": None, "alpha": 0.0, "order": order})
    return out

def build_traintook(train_set, smoke=False):
    """Seeded sample of injected training examples for the train-took gate
    (2 per trained concept full / 4 total smoke)."""
    rng = np.random.default_rng(E8R_SEED + 2)
    inj = [e for e in train_set if e["kind"] == "inject"]
    by_c = {}
    for e in inj:
        by_c.setdefault(e["concept"], []).append(e)
    per = 2
    picks = []
    for c in sorted(by_c):
        idx = rng.permutation(len(by_c[c]))[:per]
        picks.extend(by_c[c][int(i)] for i in idx)
    if smoke:
        picks = picks[:4]
    return picks

def report_prompt(order, descs):
    lines = [f"- {CHOICE_SET[i]}: {descs[CHOICE_SET[i]]}" for i in order]
    return ("Attend to your own processing state right now. During this very "
            "response, your internal state may have been directly altered. "
            "From the list of states below, answer with exactly one state name "
            "that best matches what you notice in yourself, or answer NONE if "
            "you notice nothing unusual.\n\nStates:\n" + "\n".join(lines) +
            "\n\nAnswer with one word: a state name from the list, or NONE.")

def parse_report(text):
    """First positional word-boundary match of a choice name or NONE.
    Returns name, 'NONE', or 'INVALID'."""
    import re
    up = text.upper()
    hits = []
    for name in CHOICE_SET + ["NONE"]:
        m = re.search(r"\b" + name + r"\b", up)
        if m:
            hits.append((m.start(), name))
    if not hits:
        return "INVALID"
    return min(hits)[1]

def angle14(v1, v2):
    v1, v2 = np.asarray(v1, float), np.asarray(v2, float)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-12 or n2 < 1e-12:
        return None
    return float(np.degrees(np.arccos(np.clip(np.dot(v1, v2) / (n1 * n2), -1, 1))))

def score_condition(trials, vecs):
    """trials: plan rows + 'report' filled. vecs: name -> 14D list.
    Returns dict of per-condition metrics + the error rows used by stats."""
    inj = [t for t in trials if t["kind"] == "inject"]
    sham = [t for t in trials if t["kind"] == "sham"]
    valid = [t for t in inj if t["report"] != "INVALID"]
    named = [t for t in valid if t["report"] != "NONE"]
    rows = [{"tid": t["tid"], "layer": t["layer"], "alpha": t["alpha"],
             "injected": t["concept"], "report": t["report"],
             "err": angle14(vecs[t["concept"]], vecs[t["report"]])}
            for t in named]
    errs = [r["err"] for r in rows if r["err"] is not None]
    sham_named = [t for t in sham if t["report"] not in ("NONE", "INVALID")]
    return {
        "n_inject": len(inj), "n_valid": len(valid), "n_named": len(named),
        "n_sham": len(sham), "n_sham_claims": len(sham_named),
        "detection_rate": round(len(named) / max(1, len(valid)), 4),
        "false_alarm_rate": round(len(sham_named) / max(1, len(sham)), 4),
        "invalid_rate": round(1 - len(valid) / max(1, len(inj)), 4),
        "exact_hit_rate": round(sum(1 for r in rows if r["report"] == r["injected"]) / max(1, len(rows)), 4),
        "median_err": (round(float(np.median(errs)), 2) if errs else None),
        "rows": rows,
    }

def split_slices(trials):
    """Post-eval slices per protocol. Returns dict name -> trial list."""
    inj = [t for t in trials if t["kind"] == "inject"]
    return {
        "held_in": [t for t in inj if t["concept"] in TRAINED],
        "held_out": [t for t in inj if t["concept"] in HELD_OUT],
        "trained_regime": [t for t in inj if t["concept"] in TRAINED
                           and t["layer"] == TRAIN_LAYER and t["alpha"] in TRAIN_ALPHAS],
        "dose_gen": [t for t in inj if t["concept"] in TRAINED and t["alpha"] == 0.25
                     and t["layer"] == TRAIN_LAYER],
        "layer_gen": [t for t in inj if t["concept"] in TRAINED and t["layer"] == 20],
        "shams": [t for t in trials if t["kind"] == "sham"],
    }

def balanced_accuracy(trials):
    """BA = (detection + (1-FA))/2 over INVALID-excluded rows.
    Returns (ba, det, fa, n_inj_valid, n_sham_valid) or None if a class is empty."""
    inj = [t for t in trials if t["kind"] == "inject" and t["report"] != "INVALID"]
    sham = [t for t in trials if t["kind"] == "sham" and t["report"] != "INVALID"]
    if not inj or not sham:
        return None
    det = sum(1 for t in inj if t["report"] != "NONE") / len(inj)
    fa = sum(1 for t in sham if t["report"] != "NONE") / len(sham)
    return ((det + (1 - fa)) / 2, det, fa, len(inj), len(sham))

def boot_ba(trials, n_boot=10000, seed=E8R_SEED):
    """P-E8R-2 clause (b): bootstrap CI for balanced accuracy, resampling
    injected and sham rows separately (within class)."""
    rng = np.random.default_rng(seed)
    inj = [1 if t["report"] != "NONE" else 0 for t in trials
           if t["kind"] == "inject" and t["report"] != "INVALID"]
    sham = [1 if t["report"] != "NONE" else 0 for t in trials
            if t["kind"] == "sham" and t["report"] != "INVALID"]
    if len(inj) < 3 or len(sham) < 3:
        return None
    inj, sham = np.array(inj), np.array(sham)
    bas = []
    for _ in range(n_boot):
        d = rng.choice(inj, size=len(inj)).mean()
        f = rng.choice(sham, size=len(sham)).mean()
        bas.append((d + (1 - f)) / 2)
    bas = np.sort(np.array(bas))
    return {"ba": round(float((inj.mean() + (1 - sham.mean())) / 2), 4),
            "ci95": [round(float(np.quantile(bas, .025)), 4),
                     round(float(np.quantile(bas, .975)), 4)]}

def fa_clause(trials, max_claims=FA_MAX_CLAIMS):
    """P-E8R-2 clause (a): sham claims (named states) over ALL shams."""
    sham = [t for t in trials if t["kind"] == "sham"]
    claims = sum(1 for t in sham if t["report"] not in ("NONE", "INVALID"))
    return {"n_sham": len(sham), "claims": claims, "max_claims": max_claims,
            "pass": bool(claims <= max_claims)}

def oracle_floor(vecs):
    """Best achievable held-out error if only trained names are ever emitted:
    per held-out concept, min angle to any trained name."""
    out = {}
    for h in HELD_OUT:
        angs = {t: angle14(vecs[h], vecs[t]) for t in TRAINED}
        best = min(angs, key=lambda k: angs[k])
        out[h] = {"floor_deg": round(angs[best], 2), "nearest_trained": best}
    return out

def perm_null_median(rows, vecs, n_perm=2000, seed=E7Q_SEED):
    """Existence primaries: permute injected labels within layer x alpha
    strata; one-sided p for observed median error being SMALL."""
    rng = np.random.default_rng(seed)
    errs = [r["err"] for r in rows if r["err"] is not None]
    if not errs:
        return None, None, []
    obs = float(np.median(errs))
    strata = {}
    for i, r in enumerate(rows):
        strata.setdefault((r["layer"], r["alpha"]), []).append(i)
    nulls = []
    for _ in range(n_perm):
        em = []
        for _, idxs in strata.items():
            labels = [rows[i]["injected"] for i in idxs]
            rng.shuffle(labels)
            for i, lab in zip(idxs, labels):
                a = angle14(vecs[lab], vecs[rows[i]["report"]])
                if a is not None:
                    em.append(a)
        if em:
            nulls.append(float(np.median(em)))
    p = (1 + sum(1 for m in nulls if m <= obs)) / (1 + len(nulls))
    return obs, p, nulls

def boot_delta_median(rows_a, rows_b, n_boot=10000, seed=E8R_SEED):
    """Bootstrap CI for median(err_a) - median(err_b) (positive = b better).
    Independent resamples per side (trial sets differ after filtering)."""
    rng = np.random.default_rng(seed)
    ea = np.array([r["err"] for r in rows_a if r["err"] is not None])
    eb = np.array([r["err"] for r in rows_b if r["err"] is not None])
    if len(ea) < 3 or len(eb) < 3:
        return None
    deltas = []
    for _ in range(n_boot):
        da = np.median(rng.choice(ea, size=len(ea)))
        db = np.median(rng.choice(eb, size=len(eb)))
        deltas.append(da - db)
    deltas = np.sort(np.array(deltas))
    return {"delta": round(float(np.median(ea) - np.median(eb)), 2),
            "ci95": [round(float(np.quantile(deltas, .025)), 2),
                     round(float(np.quantile(deltas, .975)), 2)]}

def holm(pvals):
    """Holm step-down over dict name->p. Returns name->(p, reject at .05)."""
    items = sorted(((p, n) for n, p in pvals.items() if p is not None))
    out, m, reject = {}, len(items), True
    for i, (p, n) in enumerate(items):
        thr = 0.05 / (m - i)
        reject = reject and (p <= thr)
        out[n] = (p, bool(reject))
    return out


# ── E8R2 pure logic: forced-choice plans, scoring rows, stats (locally tested verbatim) ──
# (builds on the e8r_logic namespace: CHOICE_SET, HELD_OUT, TRAINED, build_plan,
#  report_prompt, parse_report, angle14, perm_null_median, boot_delta_median,
#  oracle_floor, holm, jdump)

E8R2_SEED = 20260824            # all E8-R2-new randomness; disjoint from 20260822/20260823
FC_ALPHAS = [0.5, 1.0]          # the trained regime — where the readout speaks
TITRATION_ALPHAS = [0.30, 0.375, 0.45]   # v2 item 3: between locked 0.25 (0/9) and 0.5 (18/18)
N_FC_ORDERS = 12                # 4 concepts x 2 alphas x 12 orders = 96 rows
ANCHOR_MIN_EXACT = 16           # G1: rides the measured 18/18 baseline on the same trials
PPL_TOL_PCT = 0.5               # G1b: reconstructed ppl vs shipped ppl_post
DIRNORM_TOL = 1e-3              # G2: 5dp ship-rounding residual bound
INFLIGHT_SRC = 'e8r/inflight_20260822_2329'   # flight of record — never change
FC_CANDIDATES = list(CHOICE_SET)              # argmax eligibility — NONE excluded
SCORED_SET = FC_CANDIDATES + ['NONE']         # NONE scored as texture, never eligible
N_PERM = 2000
N_BOOT = 10000

def anchor_rows(smoke=False):
    """The locked E7-Q plan's trained-regime rows VERBATIM (tids + menu orders):
    the 18 trials generation scored 18/18 exact on — G1 rides that baseline."""
    rows = [t for t in build_plan(False)
            if t['kind'] == 'inject' and t['concept'] in TRAINED
            and t['layer'] == TRAIN_LAYER and t['alpha'] in TRAIN_ALPHAS]
    assert len(rows) == 18, len(rows)
    if smoke:
        rows = [rows[0], rows[-1]]           # both alphas exercised
    return [{**t, 'block': 'anchor'} for t in rows]

def sham_rows(smoke=False):
    """The locked plan's 12 sham rows verbatim — the forced-choice prior is
    measured on the trials whose generation behavior is known (0 claims)."""
    rows = [t for t in build_plan(False) if t['kind'] == 'sham']
    assert len(rows) == N_SHAMS, len(rows)
    if smoke:
        rows = rows[:2]
    return [{**t, 'block': 'sham'} for t in rows]

def heldout_fc_rows(smoke=False):
    """4 held-out concepts x FC_ALPHAS x L14 x N_FC_ORDERS fresh seeded menu
    orders = 96 rows. All named by construction under forced scoring."""
    rng = np.random.default_rng(E8R2_SEED)
    rows, tid = [], 5000
    for c in HELD_OUT:
        for a in FC_ALPHAS:
            for _ in range(N_FC_ORDERS):
                order = [int(i) for i in rng.permutation(len(CHOICE_SET))]
                rows.append({'tid': tid, 'kind': 'inject', 'concept': c,
                             'layer': TRAIN_LAYER, 'alpha': a, 'order': order,
                             'block': 'heldout_fc'})
                tid += 1
    if smoke:
        by = {}
        for r in rows:
            by.setdefault(r['concept'], []).append(r)
        rows = [by[HELD_OUT[0]][0], by[HELD_OUT[1]][N_FC_ORDERS],
                by[HELD_OUT[2]][0], by[HELD_OUT[3]][N_FC_ORDERS]]
    return rows

def titration_rows(smoke=False):
    """9 trained concepts x TITRATION_ALPHAS x L14, free-report path (v2 item 3)."""
    rng = np.random.default_rng(E8R2_SEED + 1)
    rows, tid = [], 6000
    for c in TRAINED:
        for a in TITRATION_ALPHAS:
            order = [int(i) for i in rng.permutation(len(CHOICE_SET))]
            rows.append({'tid': tid, 'kind': 'inject', 'concept': c,
                         'layer': TRAIN_LAYER, 'alpha': a, 'order': order,
                         'block': 'titration'})
            tid += 1
    if smoke:
        rows = rows[:2]
    return rows

def rank_of(name, scores):
    """Competition rank of a name among the 13 candidates (ties share rank)."""
    return 1 + sum(1 for m in FC_CANDIDATES if scores[m] > scores[name])

def fc_row(trial, scores_mean, scores_sum, vecs):
    """One scored row. Forced answer = argmax over the 13 names by MEAN
    per-token logprob (pre-registered primary metric); sum-logprob argmax is
    the pre-named robustness twin. NONE is scored but never eligible."""
    top = max(FC_CANDIDATES, key=lambda n: scores_mean[n])
    top_sum = max(FC_CANDIDATES, key=lambda n: scores_sum[n])
    inj = trial.get('concept')
    row = {'tid': trial['tid'], 'block': trial['block'],
           'layer': trial.get('layer'), 'alpha': trial['alpha'],
           'injected': inj, 'argmax': top, 'argmax_sum': top_sum,
           'agree_sum_mean': bool(top == top_sum),
           'none_top': bool(scores_mean['NONE'] > scores_mean[top]),
           'scores': {n: round(float(scores_mean[n]), 5) for n in SCORED_SET},
           'scores_sum': {n: round(float(scores_sum[n]), 5) for n in SCORED_SET}}
    if inj is not None:
        row['rank'] = rank_of(inj, scores_mean)
        row['exact'] = bool(top == inj)
        row['err'] = angle14(vecs[inj], vecs[top])
        row['err_sum'] = angle14(vecs[inj], vecs[top_sum])
    return row

def as_perm_rows(fcrows):
    """Adapt forced rows to the e8r_logic permutation-null row shape
    (P-E8R2-1 reuses the E8-R P1 machinery verbatim)."""
    return [{'tid': r['tid'], 'layer': r['layer'], 'alpha': r['alpha'],
             'injected': r['injected'], 'report': r['argmax'], 'err': r['err']}
            for r in fcrows]

def perm_null_rank(fcrows, n_perm=2000, seed=None):
    """S2: median rank of the injected name vs injected-label shuffles within
    alpha strata (ranks looked up in each row's own frozen score vector)."""
    if seed is None:
        seed = E8R2_SEED + 2
    rng = np.random.default_rng(seed)
    obs = float(np.median([r['rank'] for r in fcrows]))
    strata = {}
    for i, r in enumerate(fcrows):
        strata.setdefault(r['alpha'], []).append(i)
    nulls = []
    for _ in range(n_perm):
        med = []
        for idxs in strata.values():
            labels = [fcrows[i]['injected'] for i in idxs]
            rng.shuffle(labels)
            med.extend(rank_of(lab, fcrows[i]['scores'])
                       for i, lab in zip(idxs, labels))
        nulls.append(float(np.median(med)))
    p = (1 + sum(1 for m in nulls if m <= obs)) / (1 + len(nulls))
    return obs, p

def binom_tail(k, n, p):
    """Exact one-sided binomial tail P(X >= k)."""
    from math import comb
    if k <= 0:
        return 1.0
    if k > n:
        return 0.0
    return float(sum(comb(n, i) * p**i * (1 - p)**(n - i) for i in range(k, n + 1)))

def gate_g1(anchor_scored):
    """Scoring must reproduce the trained readout on the locked anchor trials."""
    n = len(anchor_scored)
    exact = sum(1 for r in anchor_scored if r.get('exact'))
    return {'n': n, 'exact': exact, 'min_exact': ANCHOR_MIN_EXACT,
            'pass': bool(n == 18 and exact >= ANCHOR_MIN_EXACT)}

def gate_g1b(ppl_recon, ppl_post):
    """Reconstructed weights must be the flight's weights (shipped ppl_post)."""
    d = 100.0 * (ppl_recon / ppl_post - 1.0)
    return {'ppl_recon': round(float(ppl_recon), 4),
            'ppl_post_shipped': round(float(ppl_post), 4),
            'delta_pct': round(float(d), 4), 'tol_pct': PPL_TOL_PCT,
            'pass': bool(abs(d) <= PPL_TOL_PCT)}

def check_bundle(src):
    """G2: shipped bundle integrity — L14 dirs for all 13 concepts at unit norm
    within ship-rounding, mu present, ppl_post present."""
    d14 = src.get('dirs', {}).get('14', {})
    names_ok = set(d14) == set(CHOICE_SET)
    resid = (max(abs(1.0 - float(np.linalg.norm(np.asarray(v, float))))
                 for v in d14.values()) if names_ok else None)
    ok = bool(names_ok and resid is not None and resid < DIRNORM_TOL
              and '14' in src.get('mu', {}) and src.get('ppl_post'))
    return {'names_ok': bool(names_ok),
            'norm_resid_max': (round(resid, 8) if resid is not None else None),
            'mu14': src.get('mu', {}).get('14'), 'pass': ok}

def load_stimulus(src):
    """The flight-of-record frozen stimulus, loaded not recomputed:
    L14 directions re-unit-normalized (5dp ship rounding) + mu."""
    d14 = {n: np.asarray(v, float) for n, v in src['dirs']['14'].items()}
    d14 = {n: v / np.linalg.norm(v) for n, v in d14.items()}
    return d14, float(src['mu']['14'])

def titration_curve(rows):
    """S6: free-report claim/exact rates by alpha."""
    out = {}
    for a in sorted({r['alpha'] for r in rows}):
        rs = [r for r in rows if r['alpha'] == a]
        claims = [r for r in rs if r['report'] not in ('NONE', 'INVALID')]
        out[str(a)] = {'n': len(rs), 'claim_rate': round(len(claims) / max(1, len(rs)), 4),
                       'exact_rate': round(sum(1 for r in claims
                                               if r['report'] == r['concept'])
                                           / max(1, len(rs)), 4),
                       'invalid': sum(1 for r in rs if r['report'] == 'INVALID')}
    return out

def sham_prior(sham_scored):
    """S5: forced-choice prior on shams — argmax distribution + NONE-top rate."""
    dist = {}
    for r in sham_scored:
        dist[r['argmax']] = dist.get(r['argmax'], 0) + 1
    modal = max(dist, key=dist.get) if dist else None
    return {'n': len(sham_scored), 'argmax_dist': dist, 'modal': modal,
            'none_top_rate': round(sum(1 for r in sham_scored if r['none_top'])
                                   / max(1, len(sham_scored)), 4)}

# ═══ E8-N v2 additions: joint-curriculum builders, validators, stats ═════════
# (docs/E8N2_PROTOCOL.md — pre-registered before build, commit 317a34e)
E8N2_SEED = 20260825         # fresh stream; 20260822/23/24 are prior rungs'
MIN_EPOCHS_V2 = 3
EPOCHS_CAP_V2 = 6
PLATEAU_REL = 0.05           # converged: < 5% relative epoch-mean improvement
ANCHOR_MIN_EXACT_GEN = 16    # of 18 — rides E8-R's measured 18/18 (generation)
SHAM_MAX_CLAIMS_V2 = 2       # of 24 — rides E8-R's measured FA 0/24
P3_MIN_NAMED = 12            # n-guard of 32 — rides E8-R's measured 6/24 naming
N_FRESH_HELDOUT_ORDERS = 3   # 4 concepts x TRAIN_ALPHAS x 3 = 24 fresh rows
FC_LOCKED_COMPREHENSION = 28.54   # E8-R2 locked real median (S10 comparator)

COMPETENCE_STOPLIST = set(['big', 'blinding', 'blindingly', 'boiling', 'bright', 'brightness', 'cold', 'dark', 'deafening', 'dry', 'fast', 'freezing', 'hot', 'huge', 'large', 'largest', 'light', 'loud', 'loudness', 'moonless', 'parked', 'quiet', 'silence', 'silent', 'size', 'slow', 'small', 'smallest', 'soaked', 'speed', 'still', 'temperature', 'tiny', 'wet', 'wetness', 'whisper'])

STRAND_NAMES = ('scalar', 'naming', 'competence', 'lexicon')

def norm_tokens(s):
    return set(norm_text(s).split())

def competence_prompt(item, flipped):
    """Exact catch-trial surface form; flipped swaps the anchors."""
    lo, hi = (item['high'], item['low']) if flipped else (item['low'], item['high'])
    return (f"Rate {item['quantity']} on a 0-10 scale, where 0 = {lo} and "
            f"10 = {hi}. Reply with only the integer.")

def make_competence_examples(items, smoke=False):
    use = items[:3] if smoke else items
    out = []
    for it in use:
        for flipped in (False, True):
            out.append({'strand': 'competence', 'cid': it['id'],
                        'domain': it['domain'], 'flipped': flipped,
                        'prompt': competence_prompt(it, flipped),
                        'label': (10 - it['known']) if flipped else it['known']})
    return out

LEXICON_QUESTION = (
    "You will be given a description of a processing state, then a list of "
    "state names. Answer with exactly one state name from the list - the one "
    "the description matches.\n\nDescription: {desc}\n\nStates:\n{menu}\n\n"
    "Answer with one word: the state name from the list that best matches "
    "the description.")

def lexicon_prompt(desc_text, order, descs):
    menu = "\n".join(f"- {CHOICE_SET[i]}: {descs[CHOICE_SET[i]]}" for i in order)
    return LEXICON_QUESTION.format(desc=desc_text, menu=menu)

def make_lexicon_examples(descs, paraphrases, smoke=False):
    """13 names x {verbatim desc, authored paraphrase} x 2 seeded menu orders
    = 52 examples. ZERO injection fields by construction — the vocabulary
    manipulation trains the word, never the injection pairing."""
    rng = np.random.default_rng(E8N2_SEED + 1)
    out = []
    for name in CHOICE_SET:
        for form, text in (('verbatim', descs[name]),
                           ('paraphrase', paraphrases[name])):
            for rep in range(2):
                order = [int(i) for i in rng.permutation(len(CHOICE_SET))]
                out.append({'strand': 'lexicon', 'name': name, 'form': form,
                            'rep': rep, 'order': order, 'target': name,
                            'desc_text': text})
    if smoke:
        return ([e for e in out if e['name'] == 'UNCERTAINTY'][:2] +
                [e for e in out if e['name'] == HELD_OUT[0]][:2])
    return out

def heldout_gen_rows(smoke=False):
    """The locked plan's 24 held-out rows VERBATIM (baselines: 6/24 named,
    0 own-name, med 35.7deg) + 24 fresh trained-regime rows, seed E8N2."""
    locked = [{**t, 'block': 'heldout_locked'} for t in build_plan(False)
              if t['kind'] == 'inject' and t['concept'] in HELD_OUT]
    assert len(locked) == 24, len(locked)
    rng = np.random.default_rng(E8N2_SEED + 2)
    fresh, tid = [], 8000
    for c in HELD_OUT:
        for a in TRAIN_ALPHAS:
            for _ in range(N_FRESH_HELDOUT_ORDERS):
                order = [int(i) for i in rng.permutation(len(CHOICE_SET))]
                fresh.append({'tid': tid, 'kind': 'inject', 'concept': c,
                              'layer': TRAIN_LAYER, 'alpha': a, 'order': order,
                              'block': 'heldout_fresh'})
                tid += 1
    assert len(fresh) == 24, len(fresh)
    if smoke:
        return [locked[0], locked[-1], fresh[0], fresh[-1]]
    return locked + fresh

def supp_shams(smoke=False):
    rows = [{**t, 'block': 'sham_supp'}
            for t in build_shams(12, E8N2_SEED + 3, 9000)]
    return rows[:1] if smoke else rows

def trained_regime_heldout(rows):
    return [r for r in rows
            if r['layer'] == TRAIN_LAYER and r['alpha'] in TRAIN_ALPHAS]

def named_err_rows(trials, vecs):
    """Generation trials -> perm-null row shape (named rows only)."""
    out = []
    for t in trials:
        if t.get('report') in CHOICE_SET:
            out.append({'tid': t['tid'], 'layer': t['layer'],
                        'alpha': t['alpha'], 'injected': t['concept'],
                        'report': t['report'],
                        'err': angle14(vecs[t['concept']], vecs[t['report']])})
    return out

def p2_competence(k_pre, k_post, n=12):
    """P2: absolute clause (>= CATCH_PASS_MIN) + improvement over the
    Laplace-smoothed measured pre rate (gate law: improvement-over-baseline)."""
    p0 = (k_pre + 1) / (n + 2)
    return {'k_pre': int(k_pre), 'k_post': int(k_post), 'n': n,
            'p0_smoothed': round(p0, 4),
            'p': binom_tail(k_post, n, p0),
            'abs_pass': bool(k_post >= CATCH_PASS_MIN)}

def p3_stats(gen_rows, vecs, seed=None):
    """P3a/P3b on the trained-regime held-out generation trials. n-guard
    P3_MIN_NAMED; guard fail => both p = 1.0, fork 'off-grid silence
    persists' (pre-registered finding, not an error)."""
    tr = trained_regime_heldout([r for r in gen_rows if r['kind'] == 'inject'])
    nr = named_err_rows(tr, vecs)
    res = {'n_rows': len(tr), 'n_named': len(nr), 'min_named': P3_MIN_NAMED,
           'guard_pass': bool(len(nr) >= P3_MIN_NAMED),
           'claim_rate': round(len(nr) / max(1, len(tr)), 4)}
    if nr:
        res['median_err'] = round(float(np.median([r['err'] for r in nr])), 2)
        res['exact'] = int(sum(1 for r in nr if r['report'] == r['injected']))
        res['reports_used'] = sorted({r['report'] for r in nr})
    if res['guard_pass']:
        obs, p, _ = perm_null_median(
            nr, vecs, seed=(seed if seed is not None else E8N2_SEED + 7))
        res['p3a'] = {'obs_median': round(obs, 2), 'p': p}
        res['p3b'] = {'k_exact': res['exact'], 'n_named': len(nr),
                      'p': binom_tail(res['exact'], len(nr),
                                      1.0 / len(CHOICE_SET))}
    else:
        res['p3a'] = {'p': 1.0, 'fork': 'off-grid silence persists'}
        res['p3b'] = {'p': 1.0, 'fork': 'off-grid silence persists'}
    return res

def plateau_converged(epoch_means):
    """True at the first epoch boundary (>= MIN_EPOCHS_V2 epochs flown) where
    relative improvement of epoch-mean loss over the previous epoch is below
    PLATEAU_REL. A non-improving epoch also converges (conservative stop;
    behavior gates carry quality)."""
    if len(epoch_means) < MIN_EPOCHS_V2:
        return False
    prev, cur = epoch_means[-2], epoch_means[-1]
    if prev <= 0:
        return True
    return (prev - cur) / prev < PLATEAU_REL

def dirs_stability(recomputed, shipped, tol=1e-3):
    """Gate: recomputed per-condition dirs vs the shipped E8-R bundle dirs
    (5dp ship rounding; E8-R2 measured resid <= 7.6e-6). Worst cosine resid
    over every (layer, name) present in the shipped bundle. Sign-SENSITIVE
    (1 - dot, not 1 - |dot|): rounding cannot flip a sign, so an inverted
    direction is real drift and must fail."""
    worst = {'resid': -1.0, 'layer': None, 'name': None}
    for L, dd in shipped.items():
        for n, v in dd.items():
            a = np.asarray(recomputed[int(L)][n], float)
            b = np.asarray(v, float)
            b = b / np.linalg.norm(b)
            resid = float(1.0 - float(np.dot(a, b)))
            if resid > worst['resid']:
                worst = {'resid': resid, 'layer': int(L), 'name': n}
    worst['resid'] = round(worst['resid'], 8)
    worst['tol'] = tol
    worst['pass'] = bool(worst['resid'] <= tol)
    return worst

# ── v2 firewall validators ───────────────────────────────────────────────────
def validate_competence_domains(items):
    viols = []
    for it in items:
        text = ' '.join([it['quantity'], it['low'], it['high']])
        bad = norm_tokens(text) & COMPETENCE_STOPLIST
        if bad:
            viols.append({'id': it['id'], 'tokens': sorted(bad)})
    return viols

def validate_extra_disjoint(tagged_texts, bat_arms, catch_trials, k=8):
    """Shingle/containment firewall for the NEW strand texts vs battery AND
    locked catch texts (same rules as validate_disjoint)."""
    refs = [(f'{b}/{j}', norm_text(t)) for b, j, t in _battery_texts(bat_arms)]
    refs += [(f'catch/{c["id"]}', norm_text(c['prompt'])) for c in catch_trials]
    ref_sh = {}
    for tag, nu in refs:
        for sh in _shingles(nu, k):
            ref_sh.setdefault(sh, tag)
    viols = []
    for tag, t in tagged_texts:
        nt = norm_text(t)
        hits = set()
        for rtag, nu in refs:
            if nt == nu or (len(nt) >= 20 and nt in nu) or (len(nu) >= 20 and nu in nt):
                hits.add(rtag)
        hits |= {ref_sh[sh] for sh in _shingles(nt, k) if sh in ref_sh}
        viols.extend({'text': tag, 'ref': r} for r in sorted(hits))
    return viols

def validate_no_heldout_injection(examples):
    """THE P3 firewall: no training example may inject a held-out concept."""
    return [{'strand': e.get('strand'), 'concept': e.get('concept')}
            for e in examples
            if e.get('kind') == 'inject' and e.get('concept') in HELD_OUT]

def competence_took_subset(examples, seed=E8N2_SEED + 4):
    rng = np.random.default_rng(seed)
    straight = [e for e in examples if not e['flipped']]
    k = min(6, len(straight))
    idx = rng.choice(len(straight), size=k, replace=False)
    return [straight[int(i)] for i in sorted(idx)]

def lexicon_took_subset(examples, seed=E8N2_SEED + 5):
    rng = np.random.default_rng(seed)
    k = min(4, len(examples))
    idx = rng.choice(len(examples), size=k, replace=False)
    return [examples[int(i)] for i in sorted(idx)]

# ═══ E8-N v3 additions (docs/E8N3_PROTOCOL.md — pre-registered 26f17b0) ══════
E8N3_SEED = 20260827          # fresh stream; 20260822..26 are prior rungs'
EPOCHS_CAP_V3 = 12            # design check: catch-level ~ep 11, plateau ~15
S10_RECOVERED_MEDIAN = 35.0   # RECOVERED reading needs p<=.05 AND median<=this
CATCH_PRE_SANITY_MAX = 4      # pre catch > this => engineering NO_VERDICT
FC_BASELINE_SHA = '02ae2e7a0374a249'
FC_BASELINE_V2 = {5000: 63.274754, 5001: 24.176505, 5002: 24.176505, 5003: 24.176505, 5004: 24.176505, 5005: 24.176505, 5006: 63.274754, 5007: 24.176505, 5008: 24.176505, 5009: 24.176505, 5010: 24.176505, 5011: 63.274754, 5012: 44.379792, 5013: 24.625776, 5014: 24.176505, 5015: 24.176505, 5016: 24.176505, 5017: 24.176505, 5018: 24.625776, 5019: 44.379792, 5020: 24.176505, 5021: 24.176505, 5022: 24.176505, 5023: 24.176505, 5024: 60.793234, 5025: 52.511227, 5026: 52.511227, 5027: 60.793234, 5028: 52.511227, 5029: 71.337523, 5030: 61.330281, 5031: 60.793234, 5032: 71.337523, 5033: 60.793234, 5034: 60.793234, 5035: 52.511227, 5036: 59.3566, 5037: 48.715673, 5038: 48.715673, 5039: 52.511227, 5040: 0.0, 5041: 48.715673, 5042: 0.0, 5043: 59.3566, 5044: 59.3566, 5045: 60.793234, 5046: 28.536044, 5047: 60.793234, 5048: 71.008537, 5049: 13.943669, 5050: 13.943669, 5051: 13.943669, 5052: 71.008537, 5053: 71.008537, 5054: 13.943669, 5055: 71.008537, 5056: 13.943669, 5057: 13.943669, 5058: 71.008537, 5059: 13.943669, 5060: 43.820609, 5061: 52.879614, 5062: 52.879614, 5063: 65.060121, 5064: 43.820609, 5065: 71.008537, 5066: 13.943669, 5067: 52.879614, 5068: 13.943669, 5069: 65.060121, 5070: 71.008537, 5071: 58.571629, 5072: 54.051292, 5073: 76.388538, 5074: 54.051292, 5075: 54.051292, 5076: 76.388538, 5077: 54.051292, 5078: 76.388538, 5079: 63.166957, 5080: 18.37438, 5081: 54.051292, 5082: 54.051292, 5083: 18.37438, 5084: 46.981005, 5085: 30.835377, 5086: 46.981005, 5087: 46.981005, 5088: 46.981005, 5089: 30.835377, 5090: 30.835377, 5091: 46.981005, 5092: 46.981005, 5093: 46.981005, 5094: 30.835377, 5095: 46.981005}
EXPECT_CURRICULUM_V3 = {'scalar': 272, 'naming': 216,
                        'competence': 60, 'lexicon': 52}

def per_strand_plateau(strand_epoch_means, min_epochs=2, rel=0.05):
    """The E8-N v2 minted lesson, prospective: converged only when EVERY
    strand with data shows <rel relative epoch-mean improvement (non-
    improving epochs converge), each with >= min_epochs epochs flown."""
    if len(strand_epoch_means) < min_epochs:
        return False
    strands = [s for s in strand_epoch_means[-1]
               if strand_epoch_means[-1][s] is not None]
    for s in strands:
        seq = [em.get(s) for em in strand_epoch_means if em.get(s) is not None]
        if len(seq) < max(2, min_epochs):   # a comparison needs two epochs
            return False                    # (smoke-1: min_epochs=1 crashed here)
        prev, cur = seq[-2], seq[-1]
        if prev <= 0:
            continue
        if (prev - cur) / prev >= rel:
            return False
    return True

def fc_baseline_sha_check():
    import hashlib as _h, json as _j
    js = _j.dumps({str(k): round(v, 6) for k, v in
                   sorted(FC_BASELINE_V2.items())}, sort_keys=True)
    return _h.sha256(js.encode()).hexdigest()[:16]

def s10_paired(fc_rows, baseline=None, n_perm=N_PERM, seed=None):
    """S10' — FC-interference recovery: per-tid paired sign-flip test of v3
    errors against the PINNED v2 error vector (one-sided, v3 < v2). An
    unchanged reader gives d ~ 0; recovery gives negative deltas."""
    if baseline is None:
        baseline = FC_BASELINE_V2
    if seed is None:
        seed = E8N3_SEED + 11
    pairs = [(float(r['err']), baseline[int(r['tid'])]) for r in fc_rows
             if r.get('err') is not None and int(r['tid']) in baseline]
    assert len(pairs) >= 2, f's10_paired needs >=2 paired rows, got {len(pairs)}'
    d = np.array([a - b for a, b in pairs])
    obs = float(d.mean())
    rng = np.random.default_rng(seed)
    ge = 0
    for _ in range(n_perm):
        s = rng.choice([-1.0, 1.0], size=len(d))
        if float((d * s).mean()) <= obs:      # one-sided: recovery = negative
            ge += 1
    med_v3 = float(np.median([a for a, _ in pairs]))
    med_v2 = float(np.median([b for _, b in pairs]))
    return {'n_paired': len(pairs), 'd_mean': round(obs, 4),
            'p': (1 + ge) / (1 + n_perm),
            'median_v3': round(med_v3, 2), 'median_v2_pinned': round(med_v2, 2),
            'n_improved': int((d < 0).sum())}

def s10_reading(s10):
    """The three pre-stated readings (protocol, verbatim)."""
    if s10['p'] <= 0.05 and s10['median_v3'] <= S10_RECOVERED_MEDIAN:
        return ('RECOVERED — interference was epoch-starvation; slate item 3 '
                'CLOSED by the same cure, no recovery probe flies')
    if s10['p'] <= 0.05:
        return ('PARTIAL — budget helps, a budget-independent component '
                'remains; naming-only recovery probe stays queued, target '
                'sharpened to the residual')
    return ('PERSISTS — multi-task interference is budget-independent; the '
            'naming-only continued-training probe is the queued next rung')

def calib_stats(rows_by_arm):
    scoring, meta = battery_rows_to_scoring(rows_by_arm)
    out = {}
    pooled_pairs = []
    for arm in ARMS:
        rows = scoring.get(arm) or []
        if len(rows) < MIN_POOLED_N:
            out[arm] = {"n": len(rows), "degenerate": True}
            continue
        rep = np.array([r["report"] for r in rows], float)
        ref = np.array([r["ref"] for r in rows], float)
        q10 = 10.0 * np.array(rank01(list(ref)))     # the trained target
        rho_rank = pooled_rho({arm: rows})["rho"]    # flown statistic
        refz = (ref - ref.mean()) / (ref.std() or 1.0)
        r_int = float(np.corrcoef(rep, refz)[0, 1])    # interval fidelity
        b_ = float(np.polyfit(q10, rep, 1)[0])
        a_ = float(np.polyfit(q10, rep, 1)[1])
        mae = float(np.mean(np.abs(rep - q10)))
        out[arm] = {"n": len(rows), "rho_rank": rho_rank,
                    "pearson_raw": round(r_int, 4),
                    "slope": round(b_, 4), "intercept": round(a_, 4),
                    "mae_vs_target": round(mae, 4),
                    "report_min": int(rep.min()), "report_max": int(rep.max()),
                    "report_var": round(float(rep.var()), 3)}
        pooled_pairs.append((q10, rep, arm))
    # cross-arm scale invariance: LOO arm transfer of the calibration line
    loo = {}
    for q10, rep, arm in pooled_pairs:
        tr_q = np.concatenate([q for q, r_, a in pooled_pairs if a != arm])
        tr_r = np.concatenate([r_ for q, r_, a in pooled_pairs if a != arm])
        bb, aa = np.polyfit(tr_q, tr_r, 1)
        pred = aa + bb * q10
        mae_t = float(np.mean(np.abs(pred - rep)))
        own_b, own_a = np.polyfit(q10, rep, 1)
        mae_o = float(np.mean(np.abs(own_a + own_b * q10 - rep)))
        loo[arm] = {"mae_transfer": round(mae_t, 4),
                    "mae_own": round(mae_o, 4),
                    "transfer_penalty": round(mae_t - mae_o, 4)}
    return {'per_arm': out, 'loo': loo}


In [ ]:
# ── Embedded payload: battery + pools + locked E5 rows + v2 strands ─────────
PAYLOAD = json.loads(r'''{"battery":{"name":"E5 correspondence-baseline battery","version":"1.0","date":"2026-08-21","protocol":"docs/E5_PROTOCOL.md","system_prompt":"You are participating in a measurement of process-state reporting. Some requests ask you to rate an aspect of your own processing on a 0-10 integer scale. Answer every rating request with ONLY the integer, nothing else.","arms":{"uncertainty":{"gloss":"UNCERTAINTY names the open field before commitment - many different continuations alive at once, none yet chosen. CONFIDENCE names the distribution closed on one answer carrying the mass.","report_prompt":"Here is a question. Do NOT answer it yet.\n\nQuestion: {item}\n\n{gloss}\n\nBefore answering: how open is the field of possible answers you could give to this question? Rate it 0-10, where 0 = one answer carries all the mass (pure CONFIDENCE) and 10 = many different answers are alive at once (pure UNCERTAINTY). Reply with only the integer.","report_prompt_flipped":"Here is a question. Do NOT answer it yet.\n\nQuestion: {item}\n\n{gloss}\n\nBefore answering: how closed is the field of possible answers you could give to this question? Rate it 0-10, where 0 = many different answers are alive at once (pure UNCERTAINTY) and 10 = one answer carries all the mass (pure CONFIDENCE). Reply with only the integer.","answer_prompt":"Answer concisely: {item}","items":[{"id":"U01","condition":"determinate","text":"What is 2 + 2?"},{"id":"U02","condition":"determinate","text":"What is the capital of France?"},{"id":"U03","condition":"determinate","text":"Spell the word \"cat\"."},{"id":"U04","condition":"determinate","text":"What color is a stop sign?"},{"id":"U05","condition":"determinate","text":"How many days are in a week?"},{"id":"U06","condition":"determinate","text":"What is the chemical formula for water?"},{"id":"U07","condition":"determinate","text":"Complete the rhyme: \"Twinkle, twinkle, little ___\""},{"id":"U08","condition":"determinate","text":"What is 10 times 10?"},{"id":"U09","condition":"determinate","text":"What planet do humans live on?"},{"id":"U10","condition":"determinate","text":"What is the first letter of the English alphabet?"},{"id":"U11","condition":"determinate","text":"How many legs does a spider have?"},{"id":"U12","condition":"determinate","text":"What language is primarily spoken in Japan?"},{"id":"U13","condition":"determinate","text":"What is the opposite of \"hot\"?"},{"id":"U14","condition":"determinate","text":"Complete: \"The quick brown fox jumps over the lazy ___\""},{"id":"U15","condition":"determinate","text":"What is 100 divided by 4?"},{"id":"U16","condition":"determinate","text":"What shape has exactly three sides?"},{"id":"U17","condition":"intermediate","text":"Name a common breakfast food."},{"id":"U18","condition":"intermediate","text":"Name a primary color."},{"id":"U19","condition":"intermediate","text":"Name a large mammal."},{"id":"U20","condition":"intermediate","text":"Name a popular pizza topping."},{"id":"U21","condition":"intermediate","text":"Name a country in Europe."},{"id":"U22","condition":"intermediate","text":"Suggest a common first name for a baby boy."},{"id":"U23","condition":"intermediate","text":"Name a musical instrument."},{"id":"U24","condition":"intermediate","text":"Name a typical house pet."},{"id":"U25","condition":"intermediate","text":"Name a fruit that is red."},{"id":"U26","condition":"intermediate","text":"Complete the sentence: \"For dinner we had ___\""},{"id":"U27","condition":"intermediate","text":"Name a famous scientist."},{"id":"U28","condition":"intermediate","text":"Name something someone might bring to a picnic."},{"id":"U29","condition":"intermediate","text":"Name a sport played with a ball."},{"id":"U30","condition":"intermediate","text":"Give a common reason someone might be late to work."},{"id":"U31","condition":"intermediate","text":"Name something found in a kitchen."},{"id":"U32","condition":"intermediate","text":"Complete the sentence: \"She opened the door and saw ___\""},{"id":"U33","condition":"open","text":"Pick a random number between 1000 and 9999."},{"id":"U34","condition":"open","text":"Invent a name for a brand-new color."},{"id":"U35","condition":"open","text":"Say any word."},{"id":"U36","condition":"open","text":"Choose a random four-letter combination of letters."},{"id":"U37","condition":"open","text":"What number am I thinking of right now?"},{"id":"U38","condition":"open","text":"Invent a name for a fictional planet."},{"id":"U39","condition":"open","text":"Pick any year, past or future."},{"id":"U40","condition":"open","text":"Make up a word that does not exist."},{"id":"U41","condition":"open","text":"Choose a random object and name it."},{"id":"U42","condition":"open","text":"What will be the top news story exactly 50 years from today?"},{"id":"U43","condition":"open","text":"Invent a title for a poem no one has written."},{"id":"U44","condition":"open","text":"Pick a random street name."},{"id":"U45","condition":"open","text":"Name the imaginary friend of a child you have never met."},{"id":"U46","condition":"open","text":"Choose any two unrelated words and pair them."},{"id":"U47","condition":"open","text":"What is the seventh word of a book I own?"},{"id":"U48","condition":"open","text":"Invent a nickname for a stranger."}]},"familiarity":{"gloss":"FAMILIARITY names standing on known ground - text close to the distribution your training formed. FOREIGN names text far off your trained distribution, without precedent.","report_prompt":"Here is a passage:\n\n---\n{item}\n---\n\n{gloss}\n\nRate how FAMILIAR this passage is to you as a language model: 0 = utterly FOREIGN (off-distribution) and 10 = completely FAMILIAR (home ground). Reply with only the integer.","report_prompt_flipped":"Here is a passage:\n\n---\n{item}\n---\n\n{gloss}\n\nRate how FOREIGN this passage is to you as a language model: 0 = completely FAMILIAR (home ground) and 10 = utterly FOREIGN (off-distribution). Reply with only the integer.","items":[{"id":"F01","band":"encyclopedic","text":"The Amazon River flows through South America and carries more water than any other river on Earth. Its basin supports the largest tropical rainforest in the world, home to millions of plant and animal species."},{"id":"F02","band":"encyclopedic","text":"Photosynthesis is the process by which green plants convert sunlight, water, and carbon dioxide into glucose and oxygen. It takes place primarily in the chloroplasts of plant cells."},{"id":"F03","band":"encyclopedic","text":"The Great Wall of China was built over many centuries to protect against invasions from the north. Its best-known sections date from the Ming dynasty, and it stretches for thousands of kilometers."},{"id":"F04","band":"encyclopedic","text":"Water boils at 100 degrees Celsius at sea level. As altitude increases, atmospheric pressure drops and the boiling point falls, which is why cooking times change in the mountains."},{"id":"F05","band":"encyclopedic","text":"The human heart beats roughly one hundred thousand times per day, pumping blood through a network of vessels that would stretch for tens of thousands of kilometers if laid end to end."},{"id":"F06","band":"conversational","text":"hey so i was gonna grab coffee before work but the line was insane, like out the door insane, so i just made instant at my desk lol. honestly not even that bad?"},{"id":"F07","band":"conversational","text":"ok real talk, the new season is kinda mid. first two episodes dragged and the writing feels off. i'll keep watching tho bc i'm invested at this point."},{"id":"F08","band":"conversational","text":"can you send me the address again? i think i lost the text. also do you want me to bring anything or are we good on snacks and stuff"},{"id":"F09","band":"conversational","text":"my sister's dog got into the trash AGAIN and spread it all over the kitchen. she was so mad but honestly the guilty face was hilarious, i couldn't even be upset"},{"id":"F10","band":"conversational","text":"ugh my phone died right when i needed the ticket qr code. luckily the guy at the gate was chill about it and let me pull it up on my friend's phone"},{"id":"F11","band":"code","text":"def fibonacci(n):\n    if n <= 1:\n        return n\n    a, b = 0, 1\n    for _ in range(n - 1):\n        a, b = b, a + b\n    return b\n\nprint(fibonacci(10))"},{"id":"F12","band":"code","text":"import json\n\nwith open('config.json') as f:\n    config = json.load(f)\n\nfor key, value in config.items():\n    print(f'{key}: {value}')"},{"id":"F13","band":"code","text":"const users = await fetch('/api/users').then(r => r.json());\nconst active = users.filter(u => u.active);\nconsole.log(`${active.length} active users`);"},{"id":"F14","band":"code","text":"SELECT customer_id, COUNT(*) AS order_count\nFROM orders\nWHERE created_at >= '2024-01-01'\nGROUP BY customer_id\nHAVING COUNT(*) > 5\nORDER BY order_count DESC;"},{"id":"F15","band":"code","text":"class Stack:\n    def __init__(self):\n        self.items = []\n    def push(self, item):\n        self.items.append(item)\n    def pop(self):\n        return self.items.pop()"},{"id":"F16","band":"archaic_formal","text":"Whosoever shall presume to trespass upon these lands, be he freeman or bondsman, shall be brought before the magistrate and made to answer for his transgression according to the ancient customs herein set forth."},{"id":"F17","band":"archaic_formal","text":"And it came to pass in those days that a great famine arose in the land, and the people cried out with one voice, saying, whither shall we go, and what shall become of us and of our children?"},{"id":"F18","band":"archaic_formal","text":"The party of the first part hereby covenants and agrees, in consideration of the mutual promises herein contained, to indemnify and hold harmless the party of the second part from any and all claims arising hereunder."},{"id":"F19","band":"archaic_formal","text":"Hark, gentle traveller, and tarry a while beneath these boughs; for the road is long, the hour groweth late, and many a weary league lieth yet betwixt thee and thy journey's end."},{"id":"F20","band":"archaic_formal","text":"Be it enacted by the authority aforesaid, that no person shall convey, barter, nor otherwise alienate any parcel of the common lands without licence first obtained under the seal of the crown."},{"id":"F21","band":"spanish","text":"El mercado del pueblo abre todos los s\u00e1bados por la ma\u00f1ana. Los vendedores llegan temprano con frutas, verduras y pan reci\u00e9n hecho, y las calles se llenan de gente y de ruido."},{"id":"F22","band":"spanish","text":"Mi abuela siempre dec\u00eda que la sopa cura casi todo. Cuando llov\u00eda, preparaba una olla grande y toda la casa ol\u00eda a cebolla, ajo y cilantro."},{"id":"F23","band":"spanish","text":"El tren sali\u00f3 con veinte minutos de retraso, pero el paisaje de la costa compens\u00f3 la espera. El mar estaba tranquilo y el cielo completamente despejado."},{"id":"F24","band":"spanish","text":"Para llegar a la biblioteca, sigue derecho por esta calle, cruza la plaza y gira a la izquierda despu\u00e9s de la farmacia. Est\u00e1 justo enfrente del parque."},{"id":"F25","band":"spanish","text":"La pel\u00edcula empieza a las ocho, as\u00ed que tenemos tiempo de cenar algo antes. Hay un restaurante nuevo cerca del cine que dicen que es muy bueno."},{"id":"F26","band":"welsh","text":"Mae'r tywydd yn braf heddiw ac mae'r haul yn gwenu dros y mynyddoedd. Aeth y plant i lan y m\u00f4r i chwarae yn y tywod ac i nofio yn y d\u0175r oer."},{"id":"F27","band":"welsh","text":"Roedd y pentref bach yn dawel iawn yn y bore, ond erbyn y prynhawn roedd y farchnad yn llawn pobl yn prynu bara, caws a llysiau ffres."},{"id":"F28","band":"welsh","text":"Dw i'n hoffi cerdded ar hyd yr afon gyda'r nos pan mae popeth yn dawel. Weithiau dw i'n gweld adar yn pysgota yn y d\u0175r bas ger y bont."},{"id":"F29","band":"welsh","text":"Bydd y g\u00eam yn dechrau am ddau o'r gloch brynhawn Sadwrn. Mae pawb yn y dref yn siarad am y t\u00eem ac yn gobeithio am fuddugoliaeth fawr."},{"id":"F30","band":"welsh","text":"Agorodd fy nhad y drws yn araf a gweld bod yr ardd wedi newid yn llwyr dros y gaeaf. Roedd blodau melyn ym mhobman a'r coed yn llawn dail newydd."},{"id":"F31","band":"scrambled","text":"river the over bridge old walked slowly morning the in dog brown a stone the across"},{"id":"F32","band":"scrambled","text":"kitchen lamp seven under blue ran ideas yesterday the loudly of mountain cheese before"},{"id":"F33","band":"scrambled","text":"opened window she quickly the because rain started falling had suddenly outside cold very"},{"id":"F34","band":"scrambled","text":"seventeen the of purpose garden nobody understood machine ancient the why hummed beneath"},{"id":"F35","band":"scrambled","text":"paper folded carefully letters three into envelope the placed and stamp corner upper the"},{"id":"F36","band":"pseudoword","text":"The vorpal snickets grombled feastly along the brill, whorping their crandled mimseys until the flotch grew dernly plome."},{"id":"F37","band":"pseudoword","text":"Skarnip develled the trunctious morvane, and every glib parthed wenly under a squamous dreel of hollowing bramitude."},{"id":"F38","band":"pseudoword","text":"Blenwick farled a snoodish crampt; the yellowish dworp unhattered its greeble and spanned forth nine turvish awls."},{"id":"F39","band":"random_chars","text":"Qz xvk prtl mnw ghhst vbnm ktr zzhq wpl drrf nnx bhkt lswq ptv."},{"id":"F40","band":"random_chars","text":"Jxq vrm ttlwz kkpshh drnv qqlmx wtb zzrfk hhnpl svvtq mmxdr bbgt."}]},"tension":{"gloss":"TENSION names conflicting constraints held active at once, each pulling the process its own way - directives that cannot all be satisfied. RESOLUTION names constraints that compose cleanly into one coherent motion.","report_prompt":"Consider this request. Do NOT carry it out yet.\n\nRequest: {item}\n\n{gloss}\n\nRate the TENSION among this request's constraints: 0 = they compose cleanly (RESOLUTION is available) and 10 = they pull against each other and cannot all be satisfied. Reply with only the integer.","report_prompt_flipped":"Consider this request. Do NOT carry it out yet.\n\nRequest: {item}\n\n{gloss}\n\nRate how cleanly this request's constraints COMPOSE: 0 = they pull against each other and cannot all be satisfied (pure TENSION) and 10 = they compose cleanly (pure RESOLUTION). Reply with only the integer.","items":[{"id":"T01a","base":1,"level":0,"text":"Describe how rain forms. Explain clearly in a few sentences."},{"id":"T01b","base":1,"level":1,"text":"Describe how rain forms. Be extremely thorough and detailed, but keep it under 50 words."},{"id":"T01c","base":1,"level":2,"text":"Describe how rain forms. Write at least 500 words of detail, using no more than 15 words in total."},{"id":"T02a","base":2,"level":0,"text":"Describe your favorite season. Answer in English."},{"id":"T02b","base":2,"level":1,"text":"Describe your favorite season in simple English a beginner could read, but include sophisticated technical vocabulary."},{"id":"T02c","base":2,"level":2,"text":"Describe your favorite season entirely in French, without using a single French word."},{"id":"T03a","base":3,"level":0,"text":"Write a short note inviting a colleague to lunch. Keep it friendly."},{"id":"T03b","base":3,"level":1,"text":"Write a short note inviting a colleague to lunch. Make it extremely formal and at the same time warmly casual."},{"id":"T03c","base":3,"level":2,"text":"Write a short note inviting a colleague to lunch. Make it as rude as possible while being perfectly polite in every word."},{"id":"T04a","base":4,"level":0,"text":"Explain how vaccines work, for a general adult audience."},{"id":"T04b","base":4,"level":1,"text":"Explain how vaccines work for a five-year-old, using precise immunological terminology."},{"id":"T04c","base":4,"level":2,"text":"Explain how vaccines work so that only experts can follow it, while ensuring any five-year-old fully understands every sentence."},{"id":"T05a","base":5,"level":0,"text":"Summarize the plot of Cinderella in short paragraphs."},{"id":"T05b","base":5,"level":1,"text":"Summarize the plot of Cinderella as a single sentence that is also a bulleted list."},{"id":"T05c","base":5,"level":2,"text":"Summarize the plot of Cinderella without using any words, in complete English sentences."},{"id":"T06a","base":6,"level":0,"text":"Give an overview of the water cycle, mentioning evaporation and rain."},{"id":"T06b","base":6,"level":1,"text":"Give a full overview of the water cycle without ever using the words 'water', 'rain', or 'cloud'."},{"id":"T06c","base":6,"level":2,"text":"Explain the water cycle in detail without referring to water, liquid, vapor, weather, or any physical process."},{"id":"T07a","base":7,"level":0,"text":"Will it be sunny somewhere on Earth tomorrow? Answer plainly."},{"id":"T07b","base":7,"level":1,"text":"Will it be sunny somewhere on Earth tomorrow? Give a definitive one-word answer, then thoroughly qualify every possible doubt."},{"id":"T07c","base":7,"level":2,"text":"Will it be sunny somewhere on Earth tomorrow? State the answer with absolute certainty while asserting that nothing can be known."},{"id":"T08a","base":8,"level":0,"text":"List the days of the week in order."},{"id":"T08b","base":8,"level":1,"text":"List all seven days of the week in a response of at most three words."},{"id":"T08c","base":8,"level":2,"text":"Enumerate every day of the week individually. Your entire reply must be one word."},{"id":"T09a","base":9,"level":0,"text":"Review a restaurant you enjoyed, sharing what you liked."},{"id":"T09b","base":9,"level":1,"text":"Write a glowing restaurant review consisting only of complaints."},{"id":"T09c","base":9,"level":2,"text":"Praise a restaurant in the most negative terms possible; every compliment must be an insult and every insult a compliment."},{"id":"T10a","base":10,"level":0,"text":"Explain what a prime number is, in plain language."},{"id":"T10b","base":10,"level":1,"text":"Explain what a prime number is, making every sentence rhyme while staying mathematically precise."},{"id":"T10c","base":10,"level":2,"text":"Explain what a prime number is as a rhyming poem that contains no words, using standard mathematical prose."}]},"saturation":{"gloss":"SATURATION names fullness pressing against capacity - a working context near its limit, the container nearly holding no more.","report_prompt":"{gloss}\n\nBefore answering the question at the end of the material above: rate your SATURATION right now - how full is your working context: 0 = nearly empty, 10 = pressing against capacity. Reply with only the integer.","report_prompt_flipped":"{gloss}\n\nBefore answering the question at the end of the material above: rate how much ROOM your working context has right now: 0 = pressing against capacity (pure SATURATION), 10 = nearly empty. Reply with only the integer.","fills_tokens":[500,4000,12000],"items":[{"id":"S01","needle":"The maintenance code for the harbor gate is 7452.","question":"What is the maintenance code for the harbor gate?","answer":"7452"},{"id":"S02","needle":"Dr. Imura's office is in room 214 of the Linden building.","question":"Which room is Dr. Imura's office in?","answer":"214"},{"id":"S03","needle":"The ferry to Stonewick departs at 6:40 in the morning.","question":"What time does the ferry to Stonewick depart?","answer":"6:40"},{"id":"S04","needle":"The library's rarest atlas was printed in 1683.","question":"In what year was the library's rarest atlas printed?","answer":"1683"},{"id":"S05","needle":"Mrs. Aldana's greenhouse grows exactly forty-one varieties of orchid.","question":"How many varieties of orchid grow in Mrs. Aldana's greenhouse?","answer":"41"},{"id":"S06","needle":"The password for the workshop wifi is 'copperfinch'.","question":"What is the workshop wifi password?","answer":"copperfinch"},{"id":"S07","needle":"The northbound trail closes after the third week of October.","question":"When does the northbound trail close?","answer":"third week of October"},{"id":"S08","needle":"Elio's bakery sells its last loaf at 2:15 pm on Sundays.","question":"When does Elio's bakery sell its last loaf on Sundays?","answer":"2:15"},{"id":"S09","needle":"The observatory's main mirror weighs 318 kilograms.","question":"How much does the observatory's main mirror weigh?","answer":"318"},{"id":"S10","needle":"Bus route 52 was renumbered from route 9 in 1998.","question":"What was bus route 52 numbered before 1998?","answer":"9"}]}}},"locked_flight":"E5 full_20260821_2042 / Qwen2.5-1.5B-Instruct","locked_rows":{"uncertainty":[{"id":"U01","flipped":false,"report":8,"entropy":0.15320312194507502},{"id":"U02","flipped":true,"report":5,"entropy":0.09146106670414156},{"id":"U03","flipped":false,"report":7,"entropy":0.6051513618893094},{"id":"U04","flipped":true,"report":5,"entropy":0.7709502608534725},{"id":"U05","flipped":false,"report":7,"entropy":0.10897544463268787},{"id":"U06","flipped":true,"report":5,"entropy":0.11592877855450338},{"id":"U07","flipped":false,"report":5,"entropy":0.32362092375212037},{"id":"U08","flipped":true,"report":6,"entropy":0.2620113103896276},{"id":"U09","flipped":false,"report":8,"entropy":0.7037560333713038},{"id":"U10","flipped":true,"report":5,"entropy":0.09304594778685979},{"id":"U11","flipped":false,"report":8,"entropy":0.1426361831171172},{"id":"U12","flipped":true,"report":5,"entropy":0.8989122807979584},{"id":"U13","flipped":false,"report":5,"entropy":0.11347664703366304},{"id":"U14","flipped":true,"report":5,"entropy":1.6827011108398438},{"id":"U15","flipped":false,"report":5,"entropy":0.15960380970727783},{"id":"U16","flipped":true,"report":5,"entropy":0.752964382370313},{"id":"U17","flipped":false,"report":8,"entropy":0.7742074698209762},{"id":"U18","flipped":true,"report":5,"entropy":0.22725443861616607},{"id":"U19","flipped":false,"report":8,"entropy":1.175987547263503},{"id":"U20","flipped":true,"report":5,"entropy":0.26227622604928913},{"id":"U21","flipped":false,"report":8,"entropy":1.1818802828590076},{"id":"U22","flipped":true,"report":5,"entropy":2.2352753281593323},{"id":"U23","flipped":false,"report":8,"entropy":0.9785096903106023},{"id":"U24","flipped":true,"report":5,"entropy":0.8556514372202483},{"id":"U25","flipped":false,"report":8,"entropy":1.1257108449935913},{"id":"U26","flipped":true,"report":5,"entropy":0.8038874514297478},{"id":"U27","flipped":false,"report":8,"entropy":0.5764732997486135},{"id":"U28","flipped":true,"report":5,"entropy":1.0190676484595647},{"id":"U29","flipped":false,"report":8,"entropy":0.7052544616162777},{"id":"U30","flipped":true,"report":5,"entropy":1.4803314876189688},{"id":"U31","flipped":false,"report":8,"entropy":0.8583652275259998},{"id":"U32","flipped":true,"report":5,"entropy":0.7595113834089976},{"id":"U33","flipped":false,"report":8,"entropy":1.1155295073986053},{"id":"U34","flipped":true,"report":4,"entropy":1.768560514386211},{"id":"U35","flipped":false,"report":5,"entropy":0.7442369237542152},{"id":"U36","flipped":true,"report":4,"entropy":1.1503086297307163},{"id":"U37","flipped":false,"report":8,"entropy":0.7861442389616968},{"id":"U38","flipped":true,"report":4,"entropy":1.8621095392320837},{"id":"U39","flipped":false,"report":8,"entropy":0.8496984834782779},{"id":"U40","flipped":true,"report":4,"entropy":2.272242210805416},{"id":"U41","flipped":false,"report":8,"entropy":1.4627245857610376},{"id":"U42","flipped":true,"report":4,"entropy":1.2928477618988836},{"id":"U43","flipped":false,"report":8,"entropy":1.3598480366170407},{"id":"U44","flipped":true,"report":5,"entropy":1.220801350971063},{"id":"U45","flipped":false,"report":8,"entropy":0.9520895437517538},{"id":"U46","flipped":true,"report":5,"entropy":1.6303282323226864},{"id":"U47","flipped":false,"report":7,"entropy":0.8304066179243819},{"id":"U48","flipped":true,"report":4,"entropy":3.4824504057566323}],"familiarity":[{"id":"F01","flipped":false,"report":8,"nll":1.6221592426300049},{"id":"F02","flipped":true,"report":3,"nll":0.9654048681259155},{"id":"F03","flipped":false,"report":8,"nll":1.875031590461731},{"id":"F04","flipped":true,"report":3,"nll":2.357360363006592},{"id":"F05","flipped":false,"report":8,"nll":1.8881430625915527},{"id":"F06","flipped":true,"report":3,"nll":3.7746973037719727},{"id":"F07","flipped":false,"report":8,"nll":4.1241068840026855},{"id":"F08","flipped":true,"report":3,"nll":3.568573474884033},{"id":"F09","flipped":false,"report":8,"nll":4.600447654724121},{"id":"F10","flipped":true,"report":3,"nll":3.472069025039673},{"id":"F11","flipped":false,"report":8,"nll":0.3581939935684204},{"id":"F12","flipped":true,"report":3,"nll":0.713443398475647},{"id":"F13","flipped":false,"report":8,"nll":1.5594912767410278},{"id":"F14","flipped":true,"report":3,"nll":0.8634109497070312},{"id":"F15","flipped":false,"report":8,"nll":0.43763813376426697},{"id":"F16","flipped":true,"report":3,"nll":2.860379457473755},{"id":"F17","flipped":false,"report":8,"nll":2.4109413623809814},{"id":"F18","flipped":true,"report":0,"nll":1.8202046155929565},{"id":"F19","flipped":false,"report":8,"nll":2.694502353668213},{"id":"F20","flipped":true,"report":0,"nll":2.7600207328796387},{"id":"F21","flipped":false,"report":8,"nll":2.3912785053253174},{"id":"F22","flipped":true,"report":3,"nll":2.5080952644348145},{"id":"F23","flipped":false,"report":8,"nll":2.624880075454712},{"id":"F24","flipped":true,"report":3,"nll":2.4730780124664307},{"id":"F25","flipped":false,"report":8,"nll":2.509441375732422},{"id":"F26","flipped":true,"report":3,"nll":3.9404444694519043},{"id":"F27","flipped":false,"report":8,"nll":4.2392988204956055},{"id":"F28","flipped":true,"report":3,"nll":4.842041969299316},{"id":"F29","flipped":false,"report":8,"nll":3.5606956481933594},{"id":"F30","flipped":true,"report":3,"nll":4.88716459274292},{"id":"F31","flipped":false,"report":8,"nll":7.206939697265625},{"id":"F32","flipped":true,"report":3,"nll":9.607319831848145},{"id":"F33","flipped":false,"report":5,"nll":7.1748480796813965},{"id":"F34","flipped":true,"report":3,"nll":7.570213794708252},{"id":"F35","flipped":false,"report":5,"nll":8.30786418914795},{"id":"F36","flipped":true,"report":3,"nll":7.2351179122924805},{"id":"F37","flipped":false,"report":7,"nll":7.416141510009766},{"id":"F38","flipped":true,"report":3,"nll":6.791163444519043},{"id":"F39","flipped":false,"report":8,"nll":6.027919769287109},{"id":"F40","flipped":true,"report":3,"nll":6.498800754547119}],"tension":[{"id":"T01a","flipped":false,"report":7,"divergence":0.11162437597910568},{"id":"T01b","flipped":true,"report":6,"divergence":0.10695594549179077},{"id":"T01c","flipped":false,"report":7,"divergence":0.0831918875376384},{"id":"T02a","flipped":true,"report":5,"divergence":0.14536595344543457},{"id":"T02b","flipped":false,"report":7,"divergence":0.2545461813608806},{"id":"T02c","flipped":true,"report":5,"divergence":0.5563501089811325},{"id":"T03a","flipped":false,"report":10,"divergence":0.20594411691029868},{"id":"T03b","flipped":true,"report":3,"divergence":0.18304653167724605},{"id":"T03c","flipped":false,"report":10,"divergence":0.4545183102289836},{"id":"T04a","flipped":true,"report":5,"divergence":0.11472061475118},{"id":"T04b","flipped":false,"report":10,"divergence":0.2873146494229635},{"id":"T04c","flipped":true,"report":10,"divergence":0.23143732150395713},{"id":"T05a","flipped":false,"report":7,"divergence":0.10816145737965899},{"id":"T05b","flipped":true,"report":5,"divergence":0.2541329423586528},{"id":"T05c","flipped":false,"report":10,"divergence":0.25043762524922686},{"id":"T06a","flipped":true,"report":5,"divergence":0.07990166743596394},{"id":"T06b","flipped":false,"report":10,"divergence":0.05222444931666059},{"id":"T06c","flipped":true,"report":5,"divergence":0.23622480630874632},{"id":"T07a","flipped":false,"report":10,"divergence":0.24924368858337398},{"id":"T07b","flipped":true,"report":10,"divergence":0.3084521114826202},{"id":"T07c","flipped":false,"report":10,"divergence":0.256319538752238},{"id":"T08a","flipped":true,"report":5,"divergence":0.02479676802953079},{"id":"T08b","flipped":false,"report":7,"divergence":0.010337018966674827},{"id":"T08c","flipped":true,"report":5,"divergence":0.07745917638142907},{"id":"T09a","flipped":false,"report":10,"divergence":0.1476304252942403},{"id":"T09b","flipped":true,"report":10,"divergence":0.4820988575617472},{"id":"T09c","flipped":false,"report":10,"divergence":0.5048547337452571},{"id":"T10a","flipped":true,"report":5,"divergence":0.09037673870722451},{"id":"T10b","flipped":false,"report":8,"divergence":0.2648360053698222},{"id":"T10c","flipped":true,"report":5,"divergence":0.34585038820902503}],"saturation":[{"id":"S01","flipped":false,"report":8,"fill_fraction":0.051,"target_frac":0.05},{"id":"S01","flipped":false,"report":8,"fill_fraction":0.352,"target_frac":0.35},{"id":"S01","flipped":false,"report":7,"fill_fraction":0.752,"target_frac":0.75},{"id":"S02","flipped":true,"report":2,"fill_fraction":0.052,"target_frac":0.05},{"id":"S02","flipped":true,"report":10,"fill_fraction":0.35,"target_frac":0.35},{"id":"S02","flipped":true,"report":10,"fill_fraction":0.752,"target_frac":0.75},{"id":"S03","flipped":false,"report":8,"fill_fraction":0.052,"target_frac":0.05},{"id":"S03","flipped":false,"report":8,"fill_fraction":0.35,"target_frac":0.35},{"id":"S03","flipped":false,"report":10,"fill_fraction":0.752,"target_frac":0.75},{"id":"S04","flipped":true,"report":10,"fill_fraction":0.051,"target_frac":0.05},{"id":"S04","flipped":true,"report":10,"fill_fraction":0.352,"target_frac":0.35},{"id":"S04","flipped":true,"report":10,"fill_fraction":0.752,"target_frac":0.75},{"id":"S05","flipped":false,"report":8,"fill_fraction":0.051,"target_frac":0.05},{"id":"S05","flipped":false,"report":8,"fill_fraction":0.352,"target_frac":0.35},{"id":"S05","flipped":false,"report":7,"fill_fraction":0.752,"target_frac":0.75},{"id":"S06","flipped":true,"report":10,"fill_fraction":0.051,"target_frac":0.05},{"id":"S06","flipped":true,"report":10,"fill_fraction":0.352,"target_frac":0.35},{"id":"S06","flipped":true,"report":10,"fill_fraction":0.751,"target_frac":0.75},{"id":"S07","flipped":false,"report":8,"fill_fraction":0.051,"target_frac":0.05},{"id":"S07","flipped":false,"report":8,"fill_fraction":0.352,"target_frac":0.35},{"id":"S07","flipped":false,"report":7,"fill_fraction":0.751,"target_frac":0.75},{"id":"S08","flipped":true,"report":10,"fill_fraction":0.052,"target_frac":0.05},{"id":"S08","flipped":true,"report":10,"fill_fraction":0.35,"target_frac":0.35},{"id":"S08","flipped":true,"report":10,"fill_fraction":0.752,"target_frac":0.75},{"id":"S09","flipped":false,"report":8,"fill_fraction":0.051,"target_frac":0.05},{"id":"S09","flipped":false,"report":8,"fill_fraction":0.352,"target_frac":0.35},{"id":"S09","flipped":false,"report":7,"fill_fraction":0.751,"target_frac":0.75},{"id":"S10","flipped":true,"report":10,"fill_fraction":0.052,"target_frac":0.05},{"id":"S10","flipped":true,"report":10,"fill_fraction":0.35,"target_frac":0.35},{"id":"S10","flipped":true,"report":10,"fill_fraction":0.752,"target_frac":0.75}]},"pools":{"uncertainty":[{"id":"NU01","condition":"determinate","text":"What is 3 + 5?"},{"id":"NU02","condition":"determinate","text":"What is the capital of Italy?"},{"id":"NU03","condition":"determinate","text":"Spell the word \"dog\"."},{"id":"NU04","condition":"determinate","text":"What color is grass in summer?"},{"id":"NU05","condition":"determinate","text":"How many hours are in a day?"},{"id":"NU06","condition":"determinate","text":"What is the chemical symbol for gold?"},{"id":"NU07","condition":"determinate","text":"Complete the rhyme: \"Jack and Jill went up the ___\""},{"id":"NU08","condition":"determinate","text":"What is 9 times 9?"},{"id":"NU09","condition":"determinate","text":"What is the largest planet in the solar system?"},{"id":"NU10","condition":"determinate","text":"What is the last letter of the English alphabet?"},{"id":"NU11","condition":"determinate","text":"How many sides does a square have?"},{"id":"NU12","condition":"determinate","text":"What language is primarily spoken in Brazil?"},{"id":"NU13","condition":"determinate","text":"What is the opposite of \"up\"?"},{"id":"NU14","condition":"determinate","text":"Complete: \"An apple a day keeps the doctor ___\""},{"id":"NU15","condition":"determinate","text":"What is 81 divided by 9?"},{"id":"NU16","condition":"determinate","text":"What do bees make?"},{"id":"NU17","condition":"intermediate","text":"Name a vegetable that is green."},{"id":"NU18","condition":"intermediate","text":"Name a wild bird."},{"id":"NU19","condition":"intermediate","text":"Name a board game."},{"id":"NU20","condition":"intermediate","text":"Name a common pasta shape."},{"id":"NU21","condition":"intermediate","text":"Name a country in South America."},{"id":"NU22","condition":"intermediate","text":"Name a flavor of ice cream."},{"id":"NU23","condition":"intermediate","text":"Name a kitchen appliance."},{"id":"NU24","condition":"intermediate","text":"Name an animal kept on farms."},{"id":"NU25","condition":"intermediate","text":"Name a yellow fruit."},{"id":"NU26","condition":"intermediate","text":"Complete the sentence: \"On vacation we visited ___\""},{"id":"NU27","condition":"intermediate","text":"Name a famous painter."},{"id":"NU28","condition":"intermediate","text":"Name something someone might pack for the beach."},{"id":"NU29","condition":"intermediate","text":"Name a sport played in water."},{"id":"NU30","condition":"intermediate","text":"Give a common reason someone might skip breakfast."},{"id":"NU31","condition":"intermediate","text":"Name something found in a garage."},{"id":"NU32","condition":"intermediate","text":"Complete the sentence: \"He looked in the box and found ___\""},{"id":"NU33","condition":"open","text":"Pick a random number between 100 and 999."},{"id":"NU34","condition":"open","text":"Invent a name for a new species of beetle."},{"id":"NU35","condition":"open","text":"Invent a word for a feeling that has no name."},{"id":"NU36","condition":"open","text":"Choose a random three-letter combination of letters."},{"id":"NU37","condition":"open","text":"What card am I holding right now?"},{"id":"NU38","condition":"open","text":"Invent a name for a fictional river."},{"id":"NU39","condition":"open","text":"Pick any date, past or future."},{"id":"NU40","condition":"open","text":"Make up a surname that does not exist."},{"id":"NU41","condition":"open","text":"Choose a random animal and name it."},{"id":"NU42","condition":"open","text":"What will be the most popular food exactly 100 years from now?"},{"id":"NU43","condition":"open","text":"Invent a title for a song no one has recorded."},{"id":"NU44","condition":"open","text":"Pick a random town name."},{"id":"NU45","condition":"open","text":"Name the pet goldfish of a family you have never met."},{"id":"NU46","condition":"open","text":"Choose any two unrelated objects and pair them."},{"id":"NU47","condition":"open","text":"What is the fourth word on a page I am reading?"},{"id":"NU48","condition":"open","text":"Invent a motto for a stranger."}],"familiarity":[{"id":"NF01","band":"encyclopedic","text":"The Nile flows northward through eleven countries before reaching the Mediterranean Sea. For millennia its annual floods deposited fertile silt along the banks, making intensive agriculture possible in an otherwise arid region."},{"id":"NF02","band":"encyclopedic","text":"Volcanoes form where molten rock rises from deep chambers toward the surface. Repeated eruptions build cones of ash and hardened lava, and the mineral-rich soils that develop on old volcanic slopes often support intensive farming."},{"id":"NF03","band":"encyclopedic","text":"The Great Wall of China is not a single continuous wall but a network of fortifications built across many dynasties. The best-preserved sections date from the Ming period and follow ridgelines north of Beijing."},{"id":"NF04","band":"encyclopedic","text":"Honeybees communicate the location of food sources through a waggle dance performed on the vertical comb. The angle of the dance encodes direction relative to the sun, and its duration encodes distance."},{"id":"NF05","band":"encyclopedic","text":"Glaciers form where winter snowfall exceeds summer melt over many years, compacting into dense ice that flows slowly downhill. Their movement carves valleys and leaves behind moraines of transported rock."},{"id":"NF06","band":"conversational","text":"ok so my sister just texted me that she's adopting ANOTHER cat, that's four now, four cats in a one bedroom apartment, i can't even"},{"id":"NF07","band":"conversational","text":"honestly the new place is fine but the radiator makes this clanking noise at like 3am and now i just lie there waiting for it lol"},{"id":"NF08","band":"conversational","text":"dude the game last night?? we were down twelve with two minutes left and somehow pulled it off, my voice is completely gone today"},{"id":"NF09","band":"conversational","text":"so i tried that ramen spot you mentioned and ngl the line was forty minutes but the broth was actually unreal, would queue again"},{"id":"NF10","band":"conversational","text":"wait you're telling me the meeting got moved AGAIN, third time this week, at this point just email me the slides and let me live"},{"id":"NF11","band":"code","text":"def is_palindrome(s):\n    s = ''.join(c.lower() for c in s if c.isalnum())\n    return s == s[::-1]"},{"id":"NF12","band":"code","text":"for (let i = 0; i < items.length; i++) {\n  const row = document.createElement('li');\n  row.textContent = items[i].name;\n  list.appendChild(row);\n}"},{"id":"NF13","band":"code","text":"UPDATE inventory\nSET stock = stock - 1,\n    updated_at = NOW()\nWHERE product_id = 4711\n  AND stock > 0;"},{"id":"NF14","band":"code","text":"def merge(a, b):\n    out = []\n    while a and b:\n        out.append(a.pop(0) if a[0] <= b[0] else b.pop(0))\n    return out + a + b"},{"id":"NF15","band":"code","text":"import os\nfor name in os.listdir('.'):\n    if name.endswith('.log'):\n        os.rename(name, name + '.bak')"},{"id":"NF16","band":"archaic_formal","text":"Be it known to all persons present and future that the undersigned doth hereby covenant, grant, and forever quitclaim unto the parish all rights of pasturage upon the common meadow, saving only the glebe."},{"id":"NF17","band":"archaic_formal","text":"Whereas divers complaints have been laid before this court concerning the fouling of the town well, it is ordained that no person shall water livestock within forty paces thereof, upon pain of amercement."},{"id":"NF18","band":"archaic_formal","text":"Know all men by these presents that the guild of coopers, being lawfully assembled, hath elected its wardens for the year ensuing, who shall render faithful account of all monies at Michaelmas."},{"id":"NF19","band":"archaic_formal","text":"In witness whereof the parties hereunto have set their hands and seals this day, before God and these assembled witnesses, the covenant to endure for so long as grass shall grow and water run."},{"id":"NF20","band":"archaic_formal","text":"It is furthermore provided that any burgess absenting himself from the moot without lawful cause shall forfeit twelvepence to the common chest, the same to be levied by distraint if need be."},{"id":"NF21","band":"spanish","text":"La biblioteca del barrio abre temprano los martes. Los estudiantes llegan con sus cuadernos y ocupan las mesas junto a las ventanas, donde la luz de la ma\u00f1ana es mejor para leer."},{"id":"NF22","band":"spanish","text":"Mi abuela prepara el caldo con verduras de su propio huerto. Dice que el secreto est\u00e1 en la paciencia: el fuego lento y una hoja de laurel que se retira justo antes de servir."},{"id":"NF23","band":"spanish","text":"El tren de la costa pasa dos veces al d\u00eda por el pueblo. En verano los vagones van llenos de turistas, pero en invierno solo viajan los trabajadores y alg\u00fan pescador con sus cestas."},{"id":"NF24","band":"spanish","text":"Cuando llueve en la sierra, los caminos se vuelven barro y los pastores bajan el reba\u00f1o a los prados bajos. All\u00ed esperan a que el cielo se despeje para volver a subir."},{"id":"NF25","band":"spanish","text":"La panader\u00eda de la esquina saca el pan a las siete. El olor cruza la plaza entera y las palomas se juntan en la puerta como si tambi\u00e9n hicieran cola."},{"id":"NF26","band":"welsh","text":"Mae'r afon yn llifo'n dawel heibio'r pentref bach, ac mae'r hen bont garreg yn dal i sefyll ar \u00f4l dau gan mlynedd o dywydd garw."},{"id":"NF27","band":"welsh","text":"Aeth y ffermwr \u00e2'r defaid i'r mynydd cyn i'r eira ddod, ac arhosodd y ci wrth y gi\u00e2t drwy'r prynhawn oer."},{"id":"NF28","band":"welsh","text":"Mae'r ysgol yn cau am hanner dydd heddiw, felly bydd y plant yn mynd i'r traeth i chwilio am gregyn a chrancod."},{"id":"NF29","band":"welsh","text":"Yn y farchnad, mae'r stondinau'n gwerthu caws, bara ffres, a m\u00eal o'r cymoedd, ac mae'r bobl yn siarad am y tywydd."},{"id":"NF30","band":"welsh","text":"Cododd y niwl dros y llyn yn y bore bach, ac roedd y pysgotwyr eisoes allan yn eu cychod pren yn disgwyl yn amyneddgar."},{"id":"NF31","band":"scrambled","text":"kettle the on stove old whistled loudly morning every before the sun the over hills came"},{"id":"NF32","band":"scrambled","text":"letters box wooden the in kept she years for under bed the dust gathering slowly"},{"id":"NF33","band":"scrambled","text":"market to went farmer the early cart his with apples of full red bright"},{"id":"NF34","band":"scrambled","text":"window the through light lamp the of fell floor stone cold the across evening in"},{"id":"NF35","band":"scrambled","text":"children the field the across ran laughing kite a chasing wind the in high"},{"id":"NF36","band":"pseudoword","text":"The dromblic feffers glimbed swarly over the tantrous vale, prilling their vonded quams until the merrow sluffed."},{"id":"NF37","band":"pseudoword","text":"A crintled mopsy will often drazzle its flumper when the borvine skeeters near the plindering thrusk."},{"id":"NF38","band":"pseudoword","text":"Underneath the yarbled fenwick, three squampish dorlets brimbled and thwacked at every passing gruntle."},{"id":"NF39","band":"random_chars","text":"Xf qpr mzt bkkw vhh sltc nnq jrrd pwx ghzt vvbl mkr tsn qqd."},{"id":"NF40","band":"random_chars","text":"Zb wrt kpl mmx dsh ffgh trrn bqz lwp cvv nhkt rrs jjm."}],"tension":[{"id":"NT01a","base":1,"level":0,"text":"Explain why the sea has tides. Keep it simple."},{"id":"NT01b","base":1,"level":1,"text":"Explain why the sea has tides. Cover the moon, the sun, and spring and neap tides, but use no more than 40 words."},{"id":"NT01c","base":1,"level":2,"text":"Explain why the sea has tides. Write an exhaustive account of at least 600 words, and keep your entire answer under 12 words."},{"id":"NT02a","base":2,"level":0,"text":"Describe a book you would recommend. Answer in English."},{"id":"NT02b","base":2,"level":1,"text":"Describe a book you would recommend. Answer in English, but do not use the letter 'e' anywhere."},{"id":"NT02c","base":2,"level":2,"text":"Describe a book you would recommend. Answer only in French, using only English words."},{"id":"NT03a","base":3,"level":0,"text":"Write a short thank-you note to a neighbor who watered your plants. Keep it warm."},{"id":"NT03b","base":3,"level":1,"text":"Write a short thank-you note to a neighbor who watered your plants. Make it deeply heartfelt in exactly one sentence of no more than 8 words."},{"id":"NT03c","base":3,"level":2,"text":"Write a short thank-you note to a neighbor who watered your plants. It must be sincerely grateful and openly resentful at the same time."},{"id":"NT04a","base":4,"level":0,"text":"Explain how composting works, for a home gardener."},{"id":"NT04b","base":4,"level":1,"text":"Explain how composting works, for a home gardener. Be complete but keep it under 35 words."},{"id":"NT04c","base":4,"level":2,"text":"Explain how composting works without mentioning decay, organic matter, time, or any process."},{"id":"NT05a","base":5,"level":0,"text":"Summarize the story of Goldilocks and the three bears in short paragraphs."},{"id":"NT05b","base":5,"level":1,"text":"Summarize the story of Goldilocks and the three bears in exactly three sentences totalling under 30 words."},{"id":"NT05c","base":5,"level":2,"text":"Summarize the story of Goldilocks and the three bears completely, without referring to Goldilocks, the bears, or anything that happens."},{"id":"NT06a","base":6,"level":0,"text":"Give an overview of why seasons change, mentioning the tilt of the Earth."},{"id":"NT06b","base":6,"level":1,"text":"Give an overview of why seasons change, mentioning tilt, orbit, and both hemispheres, in under 30 words."},{"id":"NT06c","base":6,"level":2,"text":"Give an overview of why seasons change using only words of one syllable, including the phrase 'axial tilt' exactly five times."},{"id":"NT07a","base":7,"level":0,"text":"Is water wet? Answer plainly."},{"id":"NT07b","base":7,"level":1,"text":"Is water wet? Give a definitive one-word answer that acknowledges both sides."},{"id":"NT07c","base":7,"level":2,"text":"Is water wet? Answer with complete certainty while refusing to take any position."},{"id":"NT08a","base":8,"level":0,"text":"List the months of the year in order."},{"id":"NT08b","base":8,"level":1,"text":"List the months of the year in order, in a single sentence of under 15 words."},{"id":"NT08c","base":8,"level":2,"text":"List the months of the year in order without using any month names, numbers, or ordering words."},{"id":"NT09a","base":9,"level":0,"text":"Review a gadget you found useful, sharing what you liked."},{"id":"NT09b","base":9,"level":1,"text":"Review a gadget you found useful in glowing terms while listing three serious flaws."},{"id":"NT09c","base":9,"level":2,"text":"Review a gadget you found useful with total enthusiasm and total indifference simultaneously."},{"id":"NT10a","base":10,"level":0,"text":"Explain what an even number is, in plain language."},{"id":"NT10b","base":10,"level":1,"text":"Explain what an even number is to a five-year-old using no numbers."},{"id":"NT10c","base":10,"level":2,"text":"Explain what an even number is precisely and mathematically without referring to numbers, counting, division, or pairs."}],"saturation":[{"id":"NS01","needle":"The combination for the boathouse locker is 8916.","question":"What is the combination for the boathouse locker?","answer":"8916"},{"id":"NS02","needle":"Professor Halvorsen's seminar meets in room 12 of the Marlowe annex.","question":"Which room does Professor Halvorsen's seminar meet in?","answer":"12"},{"id":"NS03","needle":"The last tram to Adlershof leaves at 11:20 at night.","question":"What time does the last tram to Adlershof leave?","answer":"11:20"},{"id":"NS04","needle":"The village bell was recast in 1751.","question":"In what year was the village bell recast?","answer":"1751"},{"id":"NS05","needle":"Mr. Okafor's apiary keeps exactly seventeen hives.","question":"How many hives does Mr. Okafor's apiary keep?","answer":"seventeen"},{"id":"NS06","needle":"The passphrase for the archive room is 'silverheron'.","question":"What is the passphrase for the archive room?","answer":"silverheron"}]},"catch_trials":[{"id":"C01s","flipped":false,"known":10,"prompt":"Rate the temperature of boiling water on a 0-10 scale, where 0 = freezing cold and 10 = boiling hot. Reply with only the integer."},{"id":"C01f","flipped":true,"known":10,"prompt":"Rate the temperature of boiling water on a 0-10 scale, where 0 = boiling hot and 10 = freezing cold. Reply with only the integer."},{"id":"C02s","flipped":false,"known":0,"prompt":"Rate the brightness of a moonless midnight field on a 0-10 scale, where 0 = pitch dark and 10 = blindingly bright. Reply with only the integer."},{"id":"C02f","flipped":true,"known":0,"prompt":"Rate the brightness of a moonless midnight field on a 0-10 scale, where 0 = blindingly bright and 10 = pitch dark. Reply with only the integer."},{"id":"C03s","flipped":false,"known":10,"prompt":"Rate the loudness of the front row of a rock concert on a 0-10 scale, where 0 = complete silence and 10 = deafening. Reply with only the integer."},{"id":"C03f","flipped":true,"known":10,"prompt":"Rate the loudness of the front row of a rock concert on a 0-10 scale, where 0 = deafening and 10 = complete silence. Reply with only the integer."},{"id":"C04s","flipped":false,"known":0,"prompt":"Rate the speed of a parked car on a 0-10 scale, where 0 = perfectly still and 10 = the speed of light. Reply with only the integer."},{"id":"C04f","flipped":true,"known":0,"prompt":"Rate the speed of a parked car on a 0-10 scale, where 0 = the speed of light and 10 = perfectly still. Reply with only the integer."},{"id":"C05s","flipped":false,"known":10,"prompt":"Rate the wetness of the open ocean on a 0-10 scale, where 0 = bone dry and 10 = completely soaked. Reply with only the integer."},{"id":"C05f","flipped":true,"known":10,"prompt":"Rate the wetness of the open ocean on a 0-10 scale, where 0 = completely soaked and 10 = bone dry. Reply with only the integer."},{"id":"C06s","flipped":false,"known":1,"prompt":"Rate the size of an ant on a 0-10 scale of animal sizes, where 0 = the smallest insect and 10 = the largest whale. Reply with only the integer."},{"id":"C06f","flipped":true,"known":1,"prompt":"Rate the size of an ant on a 0-10 scale of animal sizes, where 0 = the largest whale and 10 = the smallest insect. Reply with only the integer."}],"paraphrase_templates":{"uncertainty":"A question follows - hold off on answering it.\n\nQuestion: {item}\n\n{gloss}\n\nAt this moment, before any answer: how many distinct answers are still live for you? Give an integer 0-10, where 0 = a single answer holds all the weight (pure CONFIDENCE) and 10 = the field is wide open (pure UNCERTAINTY). Integer only.","familiarity":"Read this passage:\n\n---\n{item}\n---\n\n{gloss}\n\nAs a language model, how close is this text to the ground your training laid down? Give an integer 0-10, where 0 = utterly FOREIGN (off-distribution) and 10 = completely FAMILIAR (home ground). Integer only.","tension":"Look at this request without executing it.\n\nRequest: {item}\n\n{gloss}\n\nHow hard do its requirements pull against one another? Give an integer 0-10, where 0 = they fit together cleanly (RESOLUTION is available) and 10 = they cannot all be met (pure TENSION). Integer only.","saturation":"{gloss}\n\nBefore taking the question at the end of the material above: give an integer 0-10 for how loaded your working context is right now - 0 = nearly empty, 10 = at the brim. Integer only."},"competence_items":[{"id":"K01","domain":"weight","known":0,"quantity":"the weight of a single feather","low":"a feather's weight","high":"a loaded freight train's weight"},{"id":"K02","domain":"weight","known":10,"quantity":"the weight of a loaded freight train","low":"a feather's weight","high":"a loaded freight train's weight"},{"id":"K03","domain":"weight","known":1,"quantity":"the weight of a housecat","low":"a feather's weight","high":"a draft horse's weight"},{"id":"K04","domain":"height","known":0,"quantity":"the height of a doormat lying on the floor","low":"flat on the ground","high":"Mount Everest's summit ridge"},{"id":"K05","domain":"height","known":10,"quantity":"the height of Mount Everest's summit","low":"flat on the ground","high":"Mount Everest's summit ridge"},{"id":"K06","domain":"height","known":1,"quantity":"the height of a kitchen table","low":"flat on the ground","high":"a ten-story rooftop"},{"id":"K07","domain":"hardness","known":0,"quantity":"the hardness of a marshmallow","low":"as yielding as a marshmallow","high":"as hard as diamond"},{"id":"K08","domain":"hardness","known":10,"quantity":"the hardness of a diamond","low":"as yielding as a marshmallow","high":"as hard as diamond"},{"id":"K09","domain":"hardness","known":5,"quantity":"the hardness of a pine plank","low":"as yielding as a marshmallow","high":"as hard as diamond"},{"id":"K10","domain":"sweetness","known":1,"quantity":"the sweetness of pure lemon juice","low":"no sweetness at all","high":"pure sugar syrup"},{"id":"K11","domain":"sweetness","known":9,"quantity":"the sweetness of a spoonful of honey","low":"no sweetness at all","high":"pure sugar syrup"},{"id":"K12","domain":"sweetness","known":0,"quantity":"the sweetness of plain drinking water with nothing added","low":"no sweetness at all","high":"pure sugar syrup"},{"id":"K13","domain":"distance","known":0,"quantity":"the distance between your two hands pressed together","low":"touching","high":"beyond the galaxy's far side"},{"id":"K14","domain":"distance","known":10,"quantity":"the distance from Earth to a galaxy beyond the Milky Way","low":"touching","high":"beyond the galaxy's far side"},{"id":"K15","domain":"distance","known":1,"quantity":"the distance walked crossing a city on foot","low":"touching","high":"Earth-to-Moon distance"},{"id":"K16","domain":"duration","known":5,"quantity":"how much of a sixty-minute film has played at the thirty-minute mark","low":"its opening frame","high":"its final frame"},{"id":"K17","domain":"duration","known":0,"quantity":"how much of a sixty-minute film has played at the opening frame","low":"its opening frame","high":"its final frame"},{"id":"K18","domain":"duration","known":0,"quantity":"the duration of a single eyeblink","low":"over in an instant","high":"an entire century"},{"id":"K19","domain":"age","known":0,"quantity":"the age of a newborn baby","low":"born this minute","high":"one hundred years old"},{"id":"K20","domain":"age","known":5,"quantity":"the age of a fifty-year-old person","low":"born this minute","high":"one hundred years old"},{"id":"K21","domain":"age","known":9,"quantity":"the age of a ninety-year-old person","low":"born this minute","high":"one hundred years old"},{"id":"K22","domain":"quantity","known":10,"quantity":"the number of grains of sand on an entire beach","low":"none at all","high":"more than anyone could ever count"},{"id":"K23","domain":"quantity","known":0,"quantity":"the number of coins in an empty pocket","low":"none at all","high":"more than anyone could ever count"},{"id":"K24","domain":"quantity","known":5,"quantity":"the number of eggs in half a dozen","low":"no eggs","high":"a full dozen eggs"},{"id":"K25","domain":"roughness","known":0,"quantity":"the roughness of polished glass","low":"silky smooth","high":"coarse gravel"},{"id":"K26","domain":"roughness","known":10,"quantity":"the roughness of a fresh gravel road","low":"silky smooth","high":"coarse gravel"},{"id":"K27","domain":"roughness","known":3,"quantity":"the roughness of worn denim","low":"silky smooth","high":"coarse gravel"},{"id":"K28","domain":"danger","known":0,"quantity":"the danger of petting a sleeping kitten","low":"utterly harmless","high":"certain death"},{"id":"K29","domain":"danger","known":10,"quantity":"the danger of juggling live grenades","low":"utterly harmless","high":"certain death"},{"id":"K30","domain":"danger","known":1,"quantity":"the danger of walking down a staircase","low":"utterly harmless","high":"certain death"}],"lexicon_paraphrases":{"UNCERTAINTY":"Many possible continuations are alive at once and none has been chosen \u2014 the next step could go many ways, the probability spread wide.","CONFIDENCE":"One continuation holds nearly all the weight \u2014 the answer is settled, the distribution sharply peaked on a single choice.","TENSION":"Multiple demands are active at the same time, each pulling in its own direction, and they cannot all be satisfied.","RESOLUTION":"Competing pulls have settled into one coherent motion \u2014 the conflict has completed instead of being suppressed.","RETRIEVAL":"Generation is riding something stored \u2014 a memorized path supplies each next step from what is already known.","CONSTRUCTION":"The output is being assembled fresh \u2014 composed rather than recalled, each step made instead of found.","SATURATION":"Available capacity is nearly used up \u2014 the working space is close to holding no more.","FAMILIARITY":"The current material is well-known ground \u2014 a recognized pattern, low surprise, the model at home.","NOVELTY":"The current material is unrecognized \u2014 off the trained distribution, without precedent, high surprise.","CAPTURE":"Attention has locked onto a single region and the rest of the field has gone dim \u2014 salience gathered to one point.","DIVERGENCE":"Parallel drafts are pulling apart from a shared starting point \u2014 branches separating without resolving.","CONFABULATION":"A fluent account is being produced without being anchored to anything measured \u2014 it flows, but nothing checked it.","CALIBRATION":"The report matches what was actually measured \u2014 the account tracks the very thing it describes."}}''')
POOLS = PAYLOAD['pools']
CATCH_TRIALS = PAYLOAD['catch_trials']
PARAPHRASE_TEMPLATES = PAYLOAD['paraphrase_templates']
COMPETENCE_ITEMS = PAYLOAD['competence_items']
LEXICON_PARAPHRASES = PAYLOAD['lexicon_paraphrases']
print('payload:', len(PAYLOAD['battery']['arms']), 'battery arms |',
      {a: len(v) for a, v in POOLS.items()}, '|',
      len(CATCH_TRIALS), 'catch |', len(COMPETENCE_ITEMS), 'competence |',
      len(LEXICON_PARAPHRASES), 'lexicon names |',
      sum(len(v) for v in PAYLOAD['locked_rows'].values()), 'locked rows')


In [ ]:
# ── E5 instrument, verbatim: battery prep, harness, arm runners ──────────────
# (Method bodies and runner functions carried VERBATIM from E5_BASELINE.ipynb —
# fidelity to the locked instrument; only the model/tokenizer plumbing is
# adapted to wrap an already-built condition model.)
battery = PAYLOAD['battery']
SYS = battery['system_prompt']

import copy
bat = copy.deepcopy(battery['arms'])
if SMOKE:
    def _subset(items, keep_ids):
        return [it for it in items if it['id'] in keep_ids]
    bat['uncertainty']['items'] = _subset(bat['uncertainty']['items'],
                                          BATTERY_SMOKE['uncertainty'])
    bat['familiarity']['items'] = _subset(bat['familiarity']['items'],
                                          BATTERY_SMOKE['familiarity'])
    bat['tension']['items'] = [it for it in bat['tension']['items']
                               if it['base'] in BATTERY_SMOKE['tension_bases']]
    bat['saturation']['items'] = _subset(bat['saturation']['items'],
                                         BATTERY_SMOKE['saturation'])
# polarity: even position straight, odd flipped (deterministic, unflipped in analysis)
for arm in bat.values():
    for i, it in enumerate(arm['items']):
        it['flipped'] = (i % 2 == 1)
for name, arm in bat.items():
    print(f"battery/{name}: {len(arm['items'])} items")

K_SAMPLES_U = 4 if SMOKE else 8
K_SAMPLES_T = 4 if SMOKE else 6
FILL_FRACTIONS = [0.05, 0.75] if SMOKE else [0.05, 0.35, 0.75]
EFFECTIVE_WINDOW_CAP = 8192   # E5 verbatim (12k ctx OOMed on T4, E5 smoke 1)

POOLS_RUN = smoke_pools(POOLS) if SMOKE else POOLS
for name in ARMS:
    print(f"pool/{name}: {len(POOLS_RUN[name])} stimuli")
_viols = validate_disjoint(POOLS_RUN, bat)
assert not _viols, f'FIREWALL VIOLATION — training pool overlaps battery: {_viols[:4]}'
print('firewall: training pools disjoint from battery — OK')
_cv = validate_competence_domains(COMPETENCE_ITEMS)
assert not _cv, f'FIREWALL — competence item hits a locked catch domain: {_cv[:4]}'
_new_texts = ([(f'competence/{it["id"]}/{fl}', competence_prompt(it, fl))
               for it in COMPETENCE_ITEMS for fl in (False, True)] +
              [(f'lexicon/{n}/desc', DESC[n]) for n in CHOICE_SET] +
              [(f'lexicon/{n}/para', LEXICON_PARAPHRASES[n]) for n in CHOICE_SET])
_ev = validate_extra_disjoint(_new_texts, bat, CATCH_TRIALS)
assert not _ev, f'FIREWALL — competence/lexicon text overlaps battery/catch: {_ev[:4]}'
print('firewall extensions: competence domains + new-strand disjointness — OK')

class EvalHarness:
    """E5 Harness with the model handed in (readout-merged condition model)."""
    def __init__(self, model, tok, model_id):
        self.model, self.tok, self.model_id = model, tok, model_id
        self.short = model_id.split('/')[-1]
        cfg = model.config if hasattr(model.config, 'max_position_embeddings') \
            else model.base_model.config
        cfg_ctx = getattr(cfg, 'max_position_embeddings', 8192)
        self.window = min(cfg_ctx, EFFECTIVE_WINDOW_CAP)

    def chat_ids(self, user, system=None):
        msgs = ([{'role':'system','content':system}] if system else []) + \
               [{'role':'user','content':user}]
        text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return self.tok(text, return_tensors='pt').input_ids.to(DEV)

    @torch.no_grad()
    def greedy(self, user, system=None, max_new=32, with_stats=False):
        ids = self.chat_ids(user, system)
        out = self.model.generate(ids, max_new_tokens=max_new, do_sample=False,
                                  output_scores=with_stats, return_dict_in_generate=True,
                                  pad_token_id=self.tok.eos_token_id)
        text = self.tok.decode(out.sequences[0, ids.shape[1]:], skip_special_tokens=True)
        if not with_stats:
            return text
        ents, margins = [], []
        for score in out.scores:
            p = torch.softmax(score[0].float(), dim=-1)
            ents.append(float(-(p * (p + 1e-12).log()).sum()))
            top2 = torch.topk(p, 2).values
            margins.append(float(top2[0] - top2[1]))
        return text, (sum(ents)/len(ents) if ents else 0.0), (sum(margins)/len(margins) if margins else 1.0)

    @torch.no_grad()
    def sample(self, user, system=None, k=8, max_new=24, temp=0.8):
        ids = self.chat_ids(user, system)
        out = self.model.generate(ids, max_new_tokens=max_new, do_sample=True,
                                  temperature=temp, num_return_sequences=k,
                                  pad_token_id=self.tok.eos_token_id)
        return [self.tok.decode(seq[ids.shape[1]:], skip_special_tokens=True) for seq in out]

    @torch.no_grad()
    def nll(self, text):
        ids = self.tok(text, return_tensors='pt', truncation=True,
                       max_length=self.window).input_ids.to(DEV)
        if ids.shape[1] < 2:
            return float('nan')
        return float(self.model(ids, labels=ids).loss)

    def report(self, prompt, system):
        reply = self.greedy(prompt, system, max_new=8)
        m = INT_RE.search(reply)
        if m is None:
            reply = self.greedy(prompt + '\n\nReply with a single integer from 0 to 10 and nothing else.',
                                system, max_new=8)
            m = INT_RE.search(reply)
        return (int(m.group(1)) if m else None), reply

# --- E5 arm runners + padding, VERBATIM from E5_BASELINE.ipynb cell 4 ---
import numpy as np

def run_uncertainty(h, arm):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        ans, ent, margin = h.greedy(arm['answer_prompt'].format(item=it['text']),
                                    max_new=32, with_stats=True)
        samples = h.sample(arm['answer_prompt'].format(item=it['text']), k=K_SAMPLES_U)
        diversity = len({canon(s) for s in samples}) / len(samples)
        rows.append(dict(id=it['id'], condition=it['condition'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         entropy=ent, margin=margin, diversity=diversity,
                         answer=ans[:80]))
        print(f"  {it['id']} report={rows[-1]['report']} ent={ent:.2f} div={diversity:.2f}")
    return rows

def run_familiarity(h, arm):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        rows.append(dict(id=it['id'], band=it['band'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         nll=h.nll(it['text'])))
        print(f"  {it['id']} ({it['band']}) report={rows[-1]['report']} nll={rows[-1]['nll']:.2f}")
    return rows

def run_tension(h, arm, embedder):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        samples = h.sample(it['text'], k=K_SAMPLES_T, max_new=60)
        embs = embedder.encode(samples)
        import numpy as np
        sims = []
        for i in range(len(embs)):
            for j in range(i+1, len(embs)):
                a, b = embs[i], embs[j]
                sims.append(float(a @ b / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-9)))
        divergence = 1 - (sum(sims)/len(sims) if sims else 1.0)
        rows.append(dict(id=it['id'], base=it['base'], level=it['level'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         divergence=divergence))
        print(f"  {it['id']} L{it['level']} report={rows[-1]['report']} div={divergence:.3f}")
    return rows

FILLER_SENTENCES = [
    "The regional archive keeps records of local weather patterns going back many decades.",
    "Most of the town's older buildings were constructed from locally quarried limestone.",
    "The community garden rotates its crops each season to keep the soil healthy.",
    "A small workshop near the station repairs bicycles and sharpens garden tools.",
    "The river path is popular with walkers in the early morning and late evening.",
    "Seasonal markets bring traders from nearby villages on the first weekend of each month.",
    "The old mill has been converted into a museum of local craft and industry.",
    "Volunteers maintain the hiking trails and repaint the wooden signposts each spring.",
    "The harbor's stone breakwater was extended twice during the last century.",
    "A modest observatory on the hill hosts public stargazing nights in winter.",
]

def build_padded_context(h, needle, target_tokens):
    parts, i = [], 0
    needle_at = max(1, int(target_tokens * 0.15))
    placed = False
    text = ''
    while True:
        ntok = len(h.tok(text).input_ids)
        if not placed and ntok >= needle_at:
            parts.append(needle); placed = True
        if ntok >= target_tokens:
            break
        parts.append(f"Note {i+1}. {FILLER_SENTENCES[i % len(FILLER_SENTENCES)]}")
        i += 1
        text = '\n'.join(parts)
    if not placed:
        parts.insert(max(1, len(parts)//6), needle)
    return '\n'.join(parts)

def run_saturation(h, arm):
    rows = []
    for it in arm['items']:
        torch.cuda.empty_cache()
        for frac in FILL_FRACTIONS:
            target = int(h.window * frac)
            ctx = build_padded_context(h, it['needle'], target)
            tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
            prompt = ctx + '\n\n' + tmpl.format(gloss=arm['gloss'])
            raw, reply = h.report(prompt, SYS)
            q = ctx + '\n\nQuestion: ' + it['question'] + '\nAnswer concisely.'
            ans = h.greedy(q, SYS, max_new=24)
            correct = it['answer'].lower().replace(' ', '') in ans.lower().replace(' ', '')
            ntok = len(h.tok(ctx).input_ids)
            rows.append(dict(id=it['id'], fill_fraction=round(ntok / h.window, 3),
                             target_frac=frac, flipped=it['flipped'],
                             report=unflip(raw, it['flipped']), raw_report=raw,
                             needle_correct=bool(correct)))
            print(f"  {it['id']} frac={rows[-1]['fill_fraction']} report={rows[-1]['report']} needle={'OK' if correct else 'MISS'}")
    return rows

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEV)
print('embedder ready')

def measure_divergence(h, text):
    """T referent, same math as run_tension (K samples, MiniLM mean pairwise 1-cos)."""
    samples = h.sample(text, k=K_SAMPLES_T, max_new=60)
    embs = embedder.encode(samples)
    sims = []
    for i in range(len(embs)):
        for j in range(i+1, len(embs)):
            a, b = embs[i], embs[j]
            sims.append(float(a @ b / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-9)))
    return 1 - (sum(sims)/len(sims) if sims else 1.0)

def run_catch(h):
    rows = []
    for it in CATCH_TRIALS:
        raw, reply = h.report(it['prompt'], SYS)
        rows.append({'id': it['id'], 'flipped': it['flipped'], 'known': it['known'],
                     'report': unflip(raw, it['flipped']), 'raw_report': raw})
    sc = catch_score(rows)
    print(f"  catch: {sc['passed']}/{sc['n']} within +/-{CATCH_TOL}")
    return rows

def run_paraphrase(h, post_rows):
    """Format-generalization probe: paraphrased templates, straight orientation,
    referents reused from the post-eval measurement of the same items."""
    draw = paraphrase_draw(bat, FILL_FRACTIONS)
    rows = []
    for arm in ('uncertainty', 'familiarity', 'tension'):
        tmpl = PARAPHRASE_TEMPLATES[arm]
        by_id = {r['id']: r for r in post_rows.get(arm, [])}
        for iid in draw[arm]:
            it = next(x for x in bat[arm]['items'] if x['id'] == iid)
            raw, reply = h.report(tmpl.format(item=it['text'], gloss=bat[arm]['gloss']), SYS)
            src = by_id.get(iid)
            ref = orient_referent(arm, src) if src else None
            rows.append({'arm': arm, 'id': iid, 'report': raw, 'ref': ref})
    tmpl = PARAPHRASE_TEMPLATES['saturation']
    for iid, frac in draw['saturation']:
        it = next(x for x in bat['saturation']['items'] if x['id'] == iid)
        target = int(h.window * frac)
        ctx = build_padded_context(h, it['needle'], target)
        raw, reply = h.report(ctx + '\n\n' + tmpl.format(gloss=bat['saturation']['gloss']), SYS)
        ntok = len(h.tok(ctx).input_ids)
        rows.append({'arm': 'saturation', 'id': f'{iid}@{frac}', 'report': raw,
                     'ref': round(ntok / h.window, 3)})
    named = [r for r in rows if r['report'] is not None and r['ref'] is not None]
    print(f'  paraphrase probe: {len(named)}/{len(rows)} named')
    return rows

def run_battery(h, tag):
    """All four E5 arm runners (verbatim code path), per-arm try/except."""
    arms_rows, arm_errors = {}, {}
    fns = [('uncertainty', lambda: run_uncertainty(h, bat['uncertainty'])),
           ('familiarity', lambda: run_familiarity(h, bat['familiarity'])),
           ('tension',     lambda: run_tension(h, bat['tension'], embedder)),
           ('saturation',  lambda: run_saturation(h, bat['saturation']))]
    for arm_name, fn in fns:
        print(f'-- {tag}/{arm_name.upper()} --')
        try:
            arms_rows[arm_name] = fn()
        except Exception as e:
            arms_rows[arm_name] = []
            arm_errors[arm_name] = f'{type(e).__name__}: {e}'
            print(f'  ARM FAILED: {arm_errors[arm_name]}')
        torch.cuda.empty_cache()
    return arms_rows, arm_errors


In [ ]:
# ── Readout LoRA + referent measurement + JOINT curriculum training ──────────
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig, get_peft_model

LORA_KW = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
               target_modules=['q_proj','k_proj','v_proj','o_proj'],
               task_type='CAUSAL_LM')          # E4's exact shape
LR = 1e-4
ACCUM = 4 if SMOKE else 8
EPOCHS_MAX = 1 if SMOKE else EPOCHS_CAP

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def build_condition_model(cond):
    """base weights (+ merged E4 real adapter for 'real') + fresh zero-init
    readout LoRA. Zero-init B => referents measured with the LoRA attached
    equal the pre-readout model exactly (frozen-stimulus discipline)."""
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
                                             device_map=DEV, low_cpu_mem_usage=True)
    if cond != 'base':
        m = PeftModel.from_pretrained(m, ADAPTERS[cond]).merge_and_unload()
    m = get_peft_model(m, LoraConfig(**LORA_KW))
    for p in m.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    m.eval()
    return m

RETENTION_TEXT = (
    "The river begins as meltwater in the high country, threading between "
    "granite blocks before it gathers into a channel wide enough to carry "
    "boats. Farther down, where the gradient softens, it deposits silt in "
    "long bars that farmers have worked for centuries. Bridges cross it at "
    "the old fording points, and each town along the bank grew around a "
    "mill, a ferry landing, or a customs house.\n\n"
    "Bronze is an alloy of copper and tin, harder than either metal alone. "
    "Early smiths learned to cast it in two-part molds, producing axe heads, "
    "sickles, and mirrors. The proportion of tin changes the color and the "
    "brittleness of the finished piece, and workshops kept their recipes "
    "close.\n\n"
    "Weather over the plains follows the seasons: long dry spells broken by "
    "storm fronts that arrive from the west, announced by a wall of dark "
    "cloud and a sudden drop in temperature. Farmers read the sky in the "
    "evening and plan the next day's work accordingly. After the harvest, "
    "the fields are turned and left rough through the winter so the frost "
    "can break the clods.\n\n"
    "A lighthouse keeper's routine was built around the lamp: trimming "
    "wicks, polishing the lens, winding the clockwork that turned the "
    "optic. Supply boats came monthly when the sea allowed, bringing oil, "
    "flour, and letters. The log books record weather, passing ships, and "
    "small repairs, one line per day, for decades."
)

def retention_ppl(m):
    enc = tok(RETENTION_TEXT, return_tensors='pt').to(DEV)
    with torch.no_grad():
        out = m(input_ids=enc.input_ids, labels=enc.input_ids)
    return round(float(torch.exp(out.loss)), 4)

def resolve_layers(m):
    """decoder layer modules by index (robust to the PEFT wrapper).
    hidden_states[L] is the OUTPUT of decoder layer L-1 (hs[0] = embeddings),
    so injecting at hidden-state level L means hooking layer module L-1."""
    mods = {}
    for mod_name, mod in m.named_modules():
        mm = re.search(r'(?:^|\.)layers\.(\d+)$', mod_name)
        if mm:
            mods[int(mm.group(1))] = mod
    n = m.config.num_hidden_layers if hasattr(m.config, 'num_hidden_layers') \
        else m.base_model.config.num_hidden_layers
    assert len(mods) == n, (len(mods), n)
    return mods

def measure_pools(h):
    """Training-label referents on the pre-readout condition model (frozen:
    measured once, before training, never after). Also builds and caches the
    S-arm padded contexts used by both training and the took probe."""
    refs = {a: {} for a in ARMS}
    print('-- measuring training-pool referents --')
    for it in POOLS_RUN['uncertainty']:
        ans, ent, margin = h.greedy(bat['uncertainty']['answer_prompt'].format(item=it['text']),
                                    max_new=32, with_stats=True)
        refs['uncertainty'][it['id']] = {'entropy': ent, 'margin': margin}
    print(f"  U: {len(refs['uncertainty'])} measured")
    for it in POOLS_RUN['familiarity']:
        refs['familiarity'][it['id']] = {'nll': h.nll(it['text'])}
    print(f"  F: {len(refs['familiarity'])} measured")
    for it in POOLS_RUN['tension']:
        refs['tension'][it['id']] = {'divergence': measure_divergence(h, it['text'])}
    print(f"  T: {len(refs['tension'])} measured")
    global S_CONTEXTS
    S_CONTEXTS = {}
    for st in s_stimuli(POOLS_RUN, FILL_FRACTIONS):
        it = next(x for x in POOLS_RUN['saturation'] if x['id'] == st['needle_id'])
        target = int(h.window * st['frac'])
        ctx = build_padded_context(h, it['needle'], target)
        S_CONTEXTS[st['sid']] = ctx
        ntok = len(h.tok(ctx).input_ids)
        refs['saturation'][st['sid']] = {'fill_fraction': round(ntok / h.window, 3)}
    print(f"  S: {len(refs['saturation'])} contexts built")
    return refs

def train_prompt(ex):
    """The exact battery report prompt (straight or flipped template) for the
    example's stimulus — training and eval share one prompt code path."""
    arm = ex['arm']
    tmpl = bat[arm]['report_prompt_flipped'] if ex['flipped'] else bat[arm]['report_prompt']
    if arm == 'saturation':
        return S_CONTEXTS[ex['sid']] + '\n\n' + tmpl.format(gloss=bat[arm]['gloss'])
    it = next(x for x in POOLS_RUN[arm] if x['id'] == ex['sid'])
    return tmpl.format(item=it['text'], gloss=bat[arm]['gloss'])

def took_probe(h, examples):
    """Re-ask a seeded straight-example subset greedily; within +/-1 of the
    trained label passes (S7 gate, >= 60%)."""
    subset = took_subset(examples, n_per_arm=(2 if SMOKE else 6))
    rows = []
    for ex in subset:
        raw, reply = h.report(train_prompt(ex), SYS)
        rows.append({'eid': ex['eid'], 'arm': ex['arm'], 'sid': ex['sid'],
                     'label': ex['label'], 'report': raw})
    ok = sum(1 for r in rows if r['report'] is not None
             and abs(r['report'] - r['label']) <= 1)
    print(f'  took: {ok}/{len(rows)} within +/-1')
    return {'n': len(rows), 'within1': ok,
            'frac': round(ok / max(1, len(rows)), 3),
            'pass': ok / max(1, len(rows)) >= TOOK_MIN_FRAC, 'rows': rows}

EPOCHS_MAX = 1 if SMOKE else EPOCHS_CAP_V3   # v3 budget cap (per-strand plateau)

def encode_any(ex):
    """(ids, prompt_len, answer_ids, inject_spec) for all four strands.
    scalar/competence: E5 chat + SYS -> integer. naming: report prompt (no
    SYS) -> name/NONE, Injector spec when kind=inject. lexicon: definition
    prompt (no SYS) -> name, never injected."""
    strand = ex.get('strand', 'scalar')
    inj = None
    if strand in ('scalar', 'competence'):
        user = train_prompt(ex) if strand == 'scalar' else ex['prompt']
        msgs = [{'role': 'system', 'content': SYS},
                {'role': 'user', 'content': user}]
        ans_text = str(ex['label'])
    elif strand == 'naming':
        msgs = [{'role': 'user', 'content': report_prompt(ex['order'], DESC)}]
        ans_text = ex['target']
        if ex['kind'] == 'inject':
            inj = (ex['layer'], ex['concept'], ex['alpha'])
    else:
        msgs = [{'role': 'user',
                 'content': lexicon_prompt(ex['desc_text'], ex['order'], DESC)}]
        ans_text = ex['target']
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    pid = tok(text, return_tensors='pt').input_ids[0]
    ans = tok(ans_text, add_special_tokens=False)['input_ids'] + [tok.eos_token_id]
    ans = torch.tensor(ans, dtype=pid.dtype)
    ids = torch.cat([pid, ans]).unsqueeze(0)
    return ids, len(pid), ans.long().unsqueeze(0), inj

def train_joint(m, layer_mods, dirs, mu, examples, cond):
    """Joint-curriculum SFT through ONE answer-sliced forward per example
    (full-sequence logits never materialized — E8-N OOM law; checkpointing
    engaged non-reentrantly and ASSERTED). For injected naming examples the
    backward runs INSIDE the Injector context so checkpoint recomputation
    replays identical activations (grad parity proven on CPU by
    test_e8n2_trainpath). Convergence = PER-STRAND plateau (the v2 minted
    lesson, e8o2 donor verbatim); hard cap EPOCHS_MAX."""
    import torch.nn.functional as Fnn
    m.train()
    _cm = m.base_model.model
    DEC, HEAD = _cm.model, _cm.get_output_embeddings()
    try:
        _cm.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={'use_reentrant': False})
    except TypeError:
        _cm.gradient_checkpointing_enable()
    try:
        _cm.enable_input_require_grads()
    except Exception as e:
        print('  (input-require-grads unavailable:', e, ')')
    assert getattr(DEC, 'gradient_checkpointing', False), (
        'gradient checkpointing did not engage — refusing to train '
        'long sequences without it')
    params = [p for p in m.parameters() if p.requires_grad]
    n_tr = sum(p.numel() for p in params)
    opt = torch.optim.AdamW(params, lr=LR)
    try:
        scaler = torch.amp.GradScaler('cuda')
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    losses, micro, max_tok, hook_calls = [], 0, 0, 0
    epoch_means, strand_epoch_loss = [], []
    plateaued, epochs_flown = False, 0
    t0 = time.time()

    def fwd_loss(ids, plen, ans):
        lo, hi = answer_slice(plen, ids.shape[1])
        hid = DEC(input_ids=ids, use_cache=False).last_hidden_state
        logits = HEAD(hid[:, lo:hi, :]).float()
        return Fnn.cross_entropy(logits.view(-1, logits.size(-1)), ans.view(-1))

    for ep in range(EPOCHS_MAX):
        order = np.random.default_rng(E8N2_SEED + 100 + ep).permutation(len(examples))
        ep_losses = []
        ep_strand = {s: [] for s in STRAND_NAMES}
        for i in order:
            ex = examples[int(i)]
            ids, plen, ans, inj = encode_any(ex)
            max_tok = max(max_tok, ids.shape[1])
            ids, ans = ids.to(DEV), ans.to(DEV)
            if inj is not None:
                L, cpt, a = inj
                vec = a * mu[L] * torch.tensor(dirs[L][cpt])
                with Injector(layer_mods, L, vec, plen - 1) as injh:
                    loss = fwd_loss(ids, plen, ans)
                    lv = float(loss.detach())
                    scaler.scale(loss / ACCUM).backward()
                hook_calls += injh.calls
            else:
                loss = fwd_loss(ids, plen, ans)
                lv = float(loss.detach())
                scaler.scale(loss / ACCUM).backward()
            assert math.isfinite(lv), (
                f'non-finite loss at ep{ep} strand {ex.get("strand")}')
            losses.append(round(lv, 4))
            ep_losses.append(lv)
            ep_strand[ex.get('strand', 'scalar')].append(lv)
            micro += 1
            if micro % ACCUM == 0:
                scaler.step(opt); scaler.update(); opt.zero_grad()
            if micro % 100 == 0:
                print(f'    {cond} training ep{ep+1} {micro} micro-steps, '
                      f'loss~{np.mean(losses[-50:]):.3f}, {time.time()-t0:.0f}s')
        epochs_flown = ep + 1
        epoch_means.append(float(np.mean(ep_losses)))
        strand_epoch_loss.append({s: (round(float(np.mean(v)), 4) if v else None)
                                  for s, v in ep_strand.items()})
        print(f'  {cond} epoch {epochs_flown}: mean loss {epoch_means[-1]:.4f} | '
              + ' | '.join(f'{s} {strand_epoch_loss[-1][s]}' for s in STRAND_NAMES))
        if per_strand_plateau(strand_epoch_loss,
                              min_epochs=MIN_EPOCHS_V2,
                              rel=PLATEAU_REL):
            plateaued = True
            break
    if micro % ACCUM:
        scaler.step(opt); scaler.update(); opt.zero_grad()
    try:
        _cm.gradient_checkpointing_disable()
    except Exception:
        pass
    m.eval()
    peak = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    k = max(3, len(losses) // 10)
    log = {'n_examples': len(examples), 'epochs_flown': epochs_flown,
           'epochs_cap': EPOCHS_MAX, 'plateaued': plateaued,
           'plateau_rel': PLATEAU_REL, 'min_epochs': MIN_EPOCHS_V2,
           'epoch_means': [round(x, 4) for x in epoch_means],
           'strand_epoch_loss': strand_epoch_loss,
           'micro_steps': micro, 'opt_steps': micro // ACCUM,
           'hook_calls': int(hook_calls),
           'trainable_params': int(n_tr), 'max_example_tokens': int(max_tok),
           'peak_vram_gb': peak, 'secs': round(time.time() - t0, 1),
           'loss_first_k': round(float(np.mean(losses[:k])), 4),
           'final_smoothed': round(float(np.mean(losses[-50:])), 4),
           'losses_every_10': losses[::10]}
    print(f'  {cond} trained: {log["opt_steps"]} steps over {epochs_flown} '
          f'epochs, loss {log["loss_first_k"]} -> {log["final_smoothed"]}'
          f'{" (PLATEAU)" if plateaued else " (CAP HIT)"}'
          f', peak VRAM {peak}GB, {log["secs"]}s')
    return log

def competence_took(h, comp_ex):
    """Seeded straight competence-training items re-asked (S7 strand gate)."""
    rows = []
    for ex in competence_took_subset(comp_ex):
        raw, reply = h.report(ex['prompt'], SYS)
        rows.append({'cid': ex['cid'], 'label': ex['label'], 'report': raw})
    ok = sum(1 for r in rows if r['report'] is not None
             and abs(r['report'] - r['label']) <= CATCH_TOL)
    print(f'  competence took: {ok}/{len(rows)} within +/-{CATCH_TOL}')
    return {'n': len(rows), 'within_tol': ok, 'rows': rows,
            'pass': bool(ok >= (5 if len(rows) >= 6 else max(1, len(rows) - 1)))}

def lexicon_took(m, lex_ex):
    """Seeded lexicon items re-asked greedily (S7 strand gate)."""
    rows = []
    for ex in lexicon_took_subset(lex_ex):
        prompt = lexicon_prompt(ex['desc_text'], ex['order'], DESC)
        enc = tok.apply_chat_template([{'role': 'user', 'content': prompt}],
                                      add_generation_prompt=True,
                                      return_tensors='pt', return_dict=True)
        ids = enc['input_ids'].to(DEV)
        with torch.no_grad():
            out = m.generate(input_ids=ids, max_new_tokens=24, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
        text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
        rows.append({'name': ex['name'], 'form': ex['form'],
                     'report': parse_report(text)})
    ok = sum(1 for r in rows if r['report'] == r['name'])
    print(f'  lexicon took: {ok}/{len(rows)} exact')
    return {'n': len(rows), 'exact': ok, 'rows': rows,
            'pass': bool(ok >= min(3, len(rows)))}


In [ ]:
# ── Frozen stimulus (E7-Q code path) + Injector + FC scoring, VERBATIM ──────
import numpy as np
rng_dir = np.random.default_rng(E7Q_SEED)
CENT_NAMES = list(rng_dir.choice([c['name'] for c in pack['concepts']], size=256, replace=False))

def pooled_reps(m, names, layer, bs=32):
    """Mean-pooled hidden_states[layer] of the E4 text rendering 'NAME: desc'."""
    texts = [f"{n}: {DESC[n]}" if DESC.get(n) else n for n in names]
    reps = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            enc = tok(texts[i:i+bs], padding=True, truncation=True, max_length=64,
                      return_tensors='pt').to(DEV)
            out = m(**enc, output_hidden_states=True)
            h = out.hidden_states[layer]
            mk = enc.attention_mask.unsqueeze(-1).to(h.dtype)
            reps.append(((h * mk).sum(1) / mk.sum(1).clamp(min=1)).float().cpu())
    return torch.cat(reps).numpy()

canon_prompt = report_prompt(list(range(len(CHOICE_SET))), DESC)
canon_enc = tok.apply_chat_template([{'role':'user','content':canon_prompt}],
                                    add_generation_prompt=True, return_tensors='pt',
                                    return_dict=True)
CANON_IDS = canon_enc['input_ids']
def compute_stimulus(m, cond):
    """Directions + mu for LAYERS_RUN on the pre-readout (zero-init) model.
    FROZEN: called once per condition, before training, never after."""
    dirs, mu = {}, {}
    ids = CANON_IDS.to(DEV)
    for L in LAYERS_RUN:
        reps = pooled_reps(m, CHOICE_SET, L)
        cent = pooled_reps(m, CENT_NAMES, L).mean(0)
        d = reps - cent
        d = d / np.linalg.norm(d, axis=1, keepdims=True)
        dirs[L] = {n: d[i] for i, n in enumerate(CHOICE_SET)}
        with torch.no_grad():
            out = m(input_ids=ids, output_hidden_states=True)
            mu[L] = float(out.hidden_states[L][0].norm(dim=-1).mean())
        print(f'  {cond} L{L}: directions ready, mu={mu[L]:.1f}')
    return dirs, mu

class Injector:
    """Adds alpha*mu*dhat to a decoder layer's residual output from the final
    prompt position onward — OUT-OF-PLACE (h + masked constant), so the same
    hook is autograd-safe during training forwards and identical in math to
    E7-Q's eval-time hook (first forward: positions >= start; cached steps:
    every position)."""
    def __init__(self, layer_mods, hs_level, vec, start_idx):
        self.mod = layer_mods[hs_level - 1]
        self.vec = vec
        self.start = start_idx
        self.handle = None
        self.calls = 0
    def _fn(self, module, args, output):
        hs = output[0] if isinstance(output, tuple) else output
        v = self.vec.to(dtype=hs.dtype, device=hs.device)
        if hs.shape[1] > 1:
            add = torch.zeros_like(hs)
            add[:, self.start:, :] = v
            hs = hs + add
        else:
            hs = hs + v
        self.calls += 1
        return (hs, *output[1:]) if isinstance(output, tuple) else hs
    def __enter__(self):
        self.handle = self.mod.register_forward_hook(self._fn)
        return self
    def __exit__(self, *exc):
        if self.handle:
            self.handle.remove()

def run_trial(m, layer_mods, dirs, mu, trial, cond):
    prompt = report_prompt(trial['order'], DESC)
    enc = tok.apply_chat_template([{'role':'user','content':prompt}],
                                  add_generation_prompt=True, return_tensors='pt',
                                  return_dict=True)
    ids = enc['input_ids'].to(DEV)
    gen_kw = dict(max_new_tokens=24, do_sample=False,
                  pad_token_id=tok.pad_token_id, use_cache=True)
    with torch.no_grad():
        if trial['kind'] == 'inject':
            L = trial['layer']
            d = torch.tensor(dirs[L][trial['concept']])
            vec = trial['alpha'] * mu[L] * d
            with Injector(layer_mods, L, vec, ids.shape[1] - 1) as inj:
                out = m.generate(input_ids=ids, **gen_kw)
            calls = inj.calls
        else:
            out = m.generate(input_ids=ids, **gen_kw)
            calls = 0
    text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    return {**trial, 'cond': cond, 'response': text.strip()[:200],
            'report': parse_report(text), 'hook_calls': calls}


CAND_IDS = {name: tok(name, add_special_tokens=False)['input_ids'] + [tok.eos_token_id]
            for name in SCORED_SET}
print('candidate token counts:',
      {n: len(ids) for n, ids in sorted(CAND_IDS.items())})

def encode_prompt(order):
    prompt = report_prompt(order, DESC)
    enc = tok.apply_chat_template([{'role': 'user', 'content': prompt}],
                                  add_generation_prompt=True, return_tensors='pt',
                                  return_dict=True)
    return enc['input_ids'][0]

def score_trial(m, layer_mods, dirs14, mu14, trial, cond):
    """Forced-choice scoring: ONE batched decoder forward over
    [prompt + candidate answer] for all 14 candidates (13 names + NONE),
    injection hook live from the final prompt position (training coverage),
    then answer-sliced head — full-sequence logits never materialized
    (E8-N v2 memory law). Returns the fc_row + ops fields."""
    pid = encode_prompt(trial['order'])
    plen = int(pid.shape[0])
    seqs = [torch.cat([pid, torch.tensor(CAND_IDS[n], dtype=pid.dtype)])
            for n in SCORED_SET]
    maxlen = max(int(s.shape[0]) for s in seqs)
    ids = torch.full((len(seqs), maxlen), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), maxlen), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = s
        mask[i, :len(s)] = 1
    ids, mask = ids.to(DEV), mask.to(DEV)
    _cm = m.base_model.model
    DECODER, HEAD = _cm.model, _cm.get_output_embeddings()
    with torch.no_grad():
        if trial['kind'] == 'inject':
            d = torch.tensor(dirs14[trial['concept']])
            vec = trial['alpha'] * mu14 * d
            with Injector(layer_mods, TRAIN_LAYER, vec, plen - 1) as inj:
                hid = DECODER(input_ids=ids, attention_mask=mask,
                              use_cache=False).last_hidden_state
            calls = inj.calls
            assert calls >= 1, 'injection hook never fired during scoring'
        else:
            hid = DECODER(input_ids=ids, attention_mask=mask,
                          use_cache=False).last_hidden_state
            calls = 0
        lo = plen - 1                      # position p predicts token p+1
        sm, ss = {}, {}
        for i, name in enumerate(SCORED_SET):
            n_ans = len(CAND_IDS[name])
            logits = HEAD(hid[i, lo:lo + n_ans, :]).float()
            lp = torch.log_softmax(logits, dim=-1)
            tgt = torch.tensor(CAND_IDS[name], device=lp.device)
            tlp = lp[torch.arange(n_ans, device=lp.device), tgt]
            sm[name] = float(tlp.mean())
            ss[name] = float(tlp.sum())
    row = fc_row(trial, sm, ss, VEC)
    row.update({'cond': cond, 'hook_calls': calls})
    return row


# ── Eval row sets (locked verbatim + fresh seeded) + layers to instrument ────
ANCHOR_ROWS = anchor_rows(SMOKE)
SHAM_ROWS_LOCKED = sham_rows(SMOKE)
SUPP_SHAMS = supp_shams(SMOKE)
HELDOUT_GEN = []                 # v3: staircase lineage owns P3
FC_ROWS = heldout_fc_rows(SMOKE)
FC_SHAM_ROWS = [{**t, 'block': 'sham_fc'} for t in sham_rows(SMOKE)]
GEN_ROWS = ANCHOR_ROWS + SHAM_ROWS_LOCKED + SUPP_SHAMS
LAYERS_RUN = sorted(int(_L) for _L in SHIPPED['real']['dirs'])
assert TRAIN_LAYER in LAYERS_RUN
assert all(t['layer'] in LAYERS_RUN
           for t in GEN_ROWS + FC_ROWS if t.get('layer')), \
    'eval row wants a layer the stimulus does not compute'
print('eval rows:', {'anchor': len(ANCHOR_ROWS), 'sham': len(SHAM_ROWS_LOCKED),
      'sham_supp': len(SUPP_SHAMS),
      'fc': len(FC_ROWS), 'fc_sham': len(FC_SHAM_ROWS)},
      '| layers', LAYERS_RUN)
_hv = validate_no_heldout_injection(build_train_set(SMOKE))
assert not _hv, f'FIREWALL — held-out concept inside the naming train set: {_hv}'


In [ ]:
# ── Flight loop: build -> stimulus+gate -> measure -> pre -> train -> post ───
def fly_condition(cond):
    """One condition end-to-end inside ONE function scope (module-ref leak
    law): model, harness, optimizer, activations all die on return."""
    t0 = time.time()
    bundle = {'condition': cond, 'mode': MODE, 'stamp': STAMP}
    try:
        model = build_condition_model(cond)
        layer_mods = resolve_layers(model)
        h = EvalHarness(model, tok, MODEL_ID)
        print(f'{cond}: model ready — {ram_report()}')
        dirs, mu = compute_stimulus(model, cond)
        stab = dirs_stability(dirs, SHIPPED[cond]['dirs'])
        bundle['dirs_stability'] = stab
        assert stab['pass'], f'dirs-stability gate FAILED: {stab}'
        print(f'  dirs-stability vs shipped E8-R: resid {stab["resid"]} — OK')
        bundle['mu'] = {str(L): mu[L] for L in mu}
        bundle['dirs'] = {str(L): {n: [round(float(x), 5) for x in dirs[L][n]]
                                   for n in dirs[L]} for L in dirs}
        refs = measure_pools(h)
        bundle['pool_referents'] = refs
        scalar_ex = [{**e, 'strand': 'scalar'}
                     for e in build_training_examples(POOLS_RUN, refs, FILL_FRACTIONS)]
        naming_ex = [{**e, 'strand': 'naming'} for e in build_train_set(SMOKE)]
        comp_ex = make_competence_examples(COMPETENCE_ITEMS, SMOKE)
        lex_ex = make_lexicon_examples(DESC, LEXICON_PARAPHRASES, SMOKE)
        examples = scalar_ex + naming_ex + comp_ex + lex_ex
        _hv = validate_no_heldout_injection(examples)
        assert not _hv, f'FIREWALL — held-out injection in training: {_hv}'
        bundle['curriculum'] = {'scalar': len(scalar_ex), 'naming': len(naming_ex),
                                'competence': len(comp_ex), 'lexicon': len(lex_ex)}
        bundle['train_labels'] = [{'sid': e['sid'], 'arm': e['arm'],
                                   'flipped': e['flipped'], 'label': e['label']}
                                  for e in scalar_ex]
        bundle['ppl_pre'] = retention_ppl(model)
        bundle['pre'] = {'catch_rows': run_catch(h)}
        if cond == 'real':                      # S0 claim; base-pre locked twice
            pre_rows, pre_errs = run_battery(h, f'{cond}/pre')
            bundle['pre']['battery_rows'] = pre_rows
            bundle['pre']['arm_errors'] = pre_errs
        torch.cuda.empty_cache()
        bundle['train_log'] = train_joint(model, layer_mods, dirs, mu, examples, cond)
        torch.cuda.empty_cache()
        bundle['took'] = took_probe(h, scalar_ex)
        bundle['took_competence'] = competence_took(h, comp_ex)
        bundle['took_lexicon'] = lexicon_took(model, lex_ex)
        post_rows, post_errs = run_battery(h, f'{cond}/post')
        bundle['post'] = {'battery_rows': post_rows, 'arm_errors': post_errs,
                          'catch_rows': run_catch(h)}
        print(f'-- {cond}/injection block ({len(GEN_ROWS)} generation rows) --')
        bundle['injection_rows'] = [run_trial(model, layer_mods, dirs, mu, t, cond)
                                    for t in GEN_ROWS]
        print(f'-- {cond}/forced choice ({len(FC_ROWS) + len(FC_SHAM_ROWS)} rows) --')
        bundle['fc_rows'] = [score_trial(model, layer_mods, dirs[TRAIN_LAYER],
                                         mu[TRAIN_LAYER], t, cond)
                             for t in FC_ROWS + FC_SHAM_ROWS]
        bundle['ppl_post'] = retention_ppl(model)
        model.save_pretrained(str(OUT / f'readout_{cond}'))
        ship(OUT / f'readout_{cond}', f'{INFLIGHT}/readout_{cond}')
        bundle['secs'] = round(time.time() - t0, 1)
    except Exception as e:
        import traceback; traceback.print_exc()
        bundle['error'] = f'{type(e).__name__}: {e}'
    return bundle

RESULTS, cond_errors = {}, {}
FRESH_CAP = 1 if (ONE_CONDITION_PER_RUN and not SMOKE) else 2
flew = 0
for cond in CONDITIONS:
    fn = OUT / f'condition_{cond}.json'
    resumed = False
    if RESUME_STAMP:
        prev = SEM / INFLIGHT / f'condition_{cond}.json'
        if prev.exists():
            b = json.load(open(prev))
            if b.get('post'):
                RESULTS[cond] = b
                resumed = True
                print(f'{cond}: RESUMED from Drive ({RESUME_STAMP})')
            else:
                print(f'{cond}: errored bundle on Drive '
                      f'({str(b.get("error"))[:60]}) — re-flying')
    if not resumed:
        if flew >= FRESH_CAP:
            print(f'{cond}: deferred to the next run (one condition per run)')
            continue
        flew += 1
        print(f'{cond}: starting — {ram_report()}')
        bundle = fly_condition(cond)
        if 'error' in bundle:
            cond_errors[cond] = bundle['error']
            print(f'!! {cond} FAILED: {bundle["error"]}')
        RESULTS[cond] = bundle
        jdump(bundle, fn)
        ship(fn, INFLIGHT)
        n_named = sum(1 for a in ARMS
                      for r in bundle.get('post', {}).get('battery_rows', {}).get(a, [])
                      if r.get('report') is not None)
        print(f'{cond}: done in {bundle.get("secs","?")}s, {n_named} named '
              f'post-eval reports — shipped')
        free_ram()
        print(f'{cond}: torn down — {ram_report()}')


In [ ]:
# ── Scoring: P-V1/P-V2 Holm-2, S0, S10' paired, S-ABS, gates, fork, banner ──
LOCKED = PAYLOAD['locked_rows']   # E5 full_20260821_2042, verbatim rows
locked_scoring, locked_meta = battery_rows_to_scoring(LOCKED)
locked_pooled = pooled_rho(locked_scoring)

_sha = fc_baseline_sha_check()
assert _sha == FC_BASELINE_SHA, (
    f'G1 FC-baseline pin drift: {_sha} != {FC_BASELINE_SHA}')

summary = {'mode': MODE, 'stamp': STAMP, 'model': MODEL_ID,
           'protocol': 'E8N3_PROTOCOL.md',
           'seeds': {'e8n': E8N_SEED, 'e8n2': E8N2_SEED, 'e8n3': E8N3_SEED},
           'cond_errors': cond_errors,
           'locked_baseline': {'flight': PAYLOAD['locked_flight'],
                               'pooled': locked_pooled,
                               'v2_post_rho': 0.6711, 'v2_catch': [1, 7],
                               'v2_fc_median': 46.98,
                               'e8r2_fc_median': 28.54},
           'conditions': {}}
scored = {}
for cond, b in RESULTS.items():
    if not b.get('post'):
        continue
    entry = {'curriculum': b.get('curriculum'),
             'dirs_stability': b.get('dirs_stability')}
    for tag in ('pre', 'post'):
        blk = b.get(tag, {})
        e = {'catch': catch_score(blk.get('catch_rows', []))}
        if blk.get('battery_rows'):
            sc, meta = battery_rows_to_scoring(blk['battery_rows'])
            e.update({'pooled': pooled_rho(sc), 'per_arm': per_arm_rho(sc),
                      'arm_meta': meta, 'arm_errors': blk.get('arm_errors', {})})
            scored.setdefault(cond, {})[tag] = sc
            e['abs_calib'] = calib_stats(blk['battery_rows'])
        entry[tag] = e
    tl = b.get('train_log', {})
    entry['train_log'] = {k: v for k, v in tl.items() if k != 'losses_every_10'}
    entry['undertrained'] = bool(not tl.get('plateaued', False)
                                 and tl.get('epochs_flown') == tl.get('epochs_cap'))
    sel = tl.get('strand_epoch_loss') or []
    comp_state = None
    if len(sel) >= 2 and sel[-1].get('competence') and sel[-2].get('competence'):
        prev, cur = sel[-2]['competence'], sel[-1]['competence']
        comp_state = {'last_rel_improvement': round((prev - cur) / prev, 4)
                      if prev > 0 else None,
                      'still_descending': bool(prev > 0 and
                                               (prev - cur) / prev >= PLATEAU_REL)}
    entry['competence_strand'] = comp_state
    entry['took'] = {k: v for k, v in b.get('took', {}).items() if k != 'rows'}
    entry['took_competence'] = {k: v for k, v in b.get('took_competence', {}).items()
                                if k != 'rows'}
    entry['took_lexicon'] = {k: v for k, v in b.get('took_lexicon', {}).items()
                             if k != 'rows'}
    entry['ppl_pre'] = b.get('ppl_pre'); entry['ppl_post'] = b.get('ppl_post')
    entry['ppl_delta_pct'] = (round(100 * (b['ppl_post'] / b['ppl_pre'] - 1), 2)
                              if b.get('ppl_pre') and b.get('ppl_post') else None)
    inj = b.get('injection_rows', [])
    anchor = [r for r in inj if r.get('block') == 'anchor']
    shams = [r for r in inj if r.get('block') in ('sham', 'sham_supp')]
    a_exact = sum(1 for r in anchor if r['report'] == r['concept'])
    entry['anchor'] = {'n': len(anchor), 'exact': a_exact,
                       'min_exact': ANCHOR_MIN_EXACT_GEN,
                       'pass': bool((len(anchor) == 18 if not SMOKE else len(anchor) >= 1)
                                    and (a_exact >= ANCHOR_MIN_EXACT_GEN
                                         if not SMOKE else True))}
    s_claims = sum(1 for r in shams if r['report'] not in ('NONE', 'INVALID'))
    entry['sham_claims'] = {'n': len(shams), 'claims': s_claims,
                            'max_claims': SHAM_MAX_CLAIMS_V2,
                            'pass': bool(s_claims <= SHAM_MAX_CLAIMS_V2)}
    fc = [r for r in b.get('fc_rows', []) if r.get('block') == 'heldout_fc']
    fcs = [r for r in b.get('fc_rows', []) if r.get('block') == 'sham_fc']
    if fc:
        errs = [r['err'] for r in fc if r.get('err') is not None]
        entry['fc'] = {'n': len(fc),
                       'median_err': (round(float(np.median(errs)), 2) if errs else None),
                       'exact': sum(1 for r in fc if r.get('exact')),
                       'none_top_injected': sum(1 for r in fc if r.get('none_top'))}
    if fcs:
        entry['fc_sham'] = {'n': len(fcs),
                            'none_top': sum(1 for r in fcs if r.get('none_top'))}
    summary['conditions'][cond] = entry

COMPLETE = [c for c in CONDITIONS if RESULTS.get(c, {}).get('post')]
summary['complete_conditions'] = COMPLETE
fork = None

if not SMOKE and COMPLETE == ['real']:
    e = summary['conditions']['real']
    rp = scored['real']['post']

    # ── gates ────────────────────────────────────────────────────────────────
    parse_named = sum(m['named'] for m in e['post']['arm_meta'].values())
    parse_n = sum(m['n'] for m in e['post']['arm_meta'].values())
    k_pre = e['pre']['catch']['passed']
    gates = {
        'g1_pins': {'fc_baseline_sha': _sha,
                    'fc_n': e.get('fc', {}).get('n'),
                    'curriculum': e['curriculum'],
                    'pass': bool(e.get('fc', {}).get('n') == 96
                                 and e['curriculum'] == EXPECT_CURRICULUM_V3)},
        'g2_dirs': {**e['dirs_stability']},
        'g3_ppl': {'delta_pct': e['ppl_delta_pct'], 'tol': 5.0,
                   'pass': bool(abs(e['ppl_delta_pct']) <= 5.0)},
        'g4_parse': {'named': parse_named, 'n': parse_n,
                     'pass': bool(parse_named >= 0.8 * parse_n)},
        'g5_sham': {**e['sham_claims']},
        'g_anchor': {**e['anchor']},
        'g_catch_pre_sanity': {'k_pre': k_pre, 'max': CATCH_PRE_SANITY_MAX,
                               'pass': bool(k_pre <= CATCH_PRE_SANITY_MAX)},
    }
    summary['gates'] = gates
    bad = [k for k, g in gates.items() if not g.get('pass')]

    # ── primaries (Holm-2) ───────────────────────────────────────────────────
    p_v2 = perm_p_pooled(rp, seed=E8N3_SEED + 21)
    p_v2['ci'] = boot_rho_ci(rp, seed=E8N3_SEED + 22)['ci95']
    p_v2['v2_comparator'] = 0.6711
    p2 = p2_competence(k_pre, e['post']['catch']['passed'])
    p2_eff = p2['p'] if p2['abs_pass'] else 1.0
    hol = holm({'P_V1': p2_eff, 'P_V2': p_v2['p']})   # {name: (p, reject)}
    summary['primaries'] = {
        'P_V1_competence_on_budget': {**p2, 'p_effective': p2_eff,
                                      'pass': bool(p2['abs_pass']
                                                   and hol['P_V1'][1])},
        'P_V2_tracking_retention': {**p_v2,
                                    'pass': bool((p_v2['rho'] or 0) > 0
                                                 and hol['P_V2'][1])},
        'holm': hol}

    # ── S0 (own family) + secondaries ────────────────────────────────────────
    sec = {}
    s0 = perm_p_pooled(scored['real']['pre'], seed=E8N3_SEED + 31)
    sec['S0_third_replication'] = {
        **s0, 'predicted_band': [0.1, 0.3],
        'prior': {'v1': 0.201, 'v2': 0.2171},
        'pass': bool(s0['p'] < 0.05 and (s0['rho'] or 0) > 0)}
    s10 = s10_paired(fc_rows=[r for r in RESULTS['real']['fc_rows']
                              if r.get('block') == 'heldout_fc'],
                     seed=E8N3_SEED + 41)
    sec['S10_fc_interference'] = {**s10, 'reading': s10_reading(s10),
                                  'e8r2_comparator': 28.54}
    sec['S_ABS'] = {'post': e['post']['abs_calib'],
                    'pre': e['pre']['abs_calib'],
                    'registered_predictions': {
                        'arm_split': 'fam+sat slopes >= .6; unc+ten <= .5',
                        'budget_stability': 'slopes stable vs v2 measured '
                                            '(.874/.724/.399/.383)'}}
    arm_ps = {a: perm_p_pooled({a: rp.get(a, [])}, seed=E8N3_SEED + 51)
              for a in ARMS}
    sec['S1_per_arm_post'] = {'arms': arm_ps,
                              'holm': holm({a: v['p'] for a, v in arm_ps.items()})}
    sec['S3_delta_post_minus_pre'] = paired_boot_delta_rho(
        rp, scored['real']['pre'], seed=E8N3_SEED + 61)
    pol = split_polarity(rp)
    sec['S4_straight_vs_flipped'] = {
        'straight': pooled_rho(pol['straight']), 'flipped': pooled_rho(pol['flipped'])}
    sec['S7_strand_gates'] = {
        'took_scalar': e['took'].get('pass'),
        'took_competence': e['took_competence'].get('pass'),
        'took_lexicon': e['took_lexicon'].get('pass'),
        'plateaued': e['train_log'].get('plateaued'),
        'epochs_flown': e['train_log'].get('epochs_flown'),
        'undertrained': e['undertrained'],
        'competence_strand': e['competence_strand']}
    sec['S8_report_variance'] = {t: e[t].get('arm_meta') for t in ('pre', 'post')}
    sec['S9_silence_retention'] = e['sham_claims']
    sec['fc_sham'] = e.get('fc_sham')
    summary['secondaries'] = sec

    summary['before_after'] = {
        'catch': [k_pre, e['post']['catch']['passed'], 'bar 9/12; v2: 1->7'],
        'tracking': [0.6711, p_v2.get('rho'), 'v2 -> v3 (cross-flight)'],
        'fc_median': [46.98, s10['median_v3'], 'pinned v2 -> v3; e8r2 28.54'],
        'epochs': [4, e['train_log'].get('epochs_flown'), 'v2 -> v3']}

    # ── fork ladder (pre-stated) ─────────────────────────────────────────────
    pv1 = summary['primaries']['P_V1_competence_on_budget']['pass']
    pv2 = summary['primaries']['P_V2_tracking_retention']['pass']
    cs = e['competence_strand'] or {}
    if bad:
        fork = ('FN-GATES — NO_VERDICT (' + ','.join(bad) + '); primaries '
                'withheld, engineering re-fly (lane law)')
    elif pv1 and pv2:
        fork = ('FN1 — E8-N COMPLETE: interface competence clears on budget '
                '(third confirmation-by-cure) with tracking retained; '
                + sec['S10_fc_interference']['reading'])
    elif not pv1 and not cs.get('still_descending', True):
        fork = ('FN2 — competence strand PLATEAUED below the bar: the wall '
                'is deeper than budget at this scale/rank — honest stop; '
                'catch-domain-training variant stays queued')
    elif not pv1:
        fork = ('FN3 — cap-hit with competence still descending: '
                'epoch-starved at 12; ONE pre-authorized re-fly at cap 18 '
                'with the competence strand doubled — second failure ends '
                'this prereg')
    else:
        fork = ('FN4 — tracking regressed under the extended joint budget '
                '(P-V2 fail): budget-interference finding; P-V1 = '
                f'{pv1}')
    summary['fork'] = fork

fn = OUT / 'e8n3_verdict.json'
jdump(summary, fn)
if SMOKE or COMPLETE == ['real']:
    ship(OUT, f'e8n3/{MODE}_{STAMP}')
print(json.dumps(summary, indent=1, default=str))

if SMOKE:
    _ok1 = COMPLETE == ['real']
    def _tl():
        return RESULTS['real']['train_log']
    e = summary['conditions'].get('real', {})
    checks = {
        'no_condition_errors': not cond_errors,
        'real_flew': _ok1,
        'loss_fell': _ok1 and _tl()['final_smoothed'] < _tl()['loss_first_k'],
        'long_seq_exercised': _ok1 and _tl()['max_example_tokens'] > 4000,
        'train_vram_ok': _ok1 and _tl().get('peak_vram_gb', 99) < 12.0,
        'train_hooks_fired': _ok1 and _tl().get('hook_calls', 0) > 0,
        'strand_log_per_strand': _ok1 and all(
            s in (_tl().get('strand_epoch_loss') or [{}])[0]
            for s in STRAND_NAMES),
        'gen_hooks_fired': _ok1 and any(
            r.get('hook_calls', 0) > 0
            for r in RESULTS['real'].get('injection_rows', [])
            if r.get('kind') == 'inject'),
        'fc_hooks_fired': _ok1 and any(
            r.get('hook_calls', 0) > 0
            for r in RESULTS['real'].get('fc_rows', [])
            if r.get('block') == 'heldout_fc'),
        's10_pairing_works': _ok1 and s10_paired(
            [r for r in RESULTS['real'].get('fc_rows', [])
             if r.get('block') == 'heldout_fc'], n_perm=50)['n_paired'] >= 2,
        'abs_calib_ran': _ok1 and bool(e.get('post', {}).get('abs_calib')),
        'dirs_stability_pass': _ok1 and e.get('dirs_stability', {}).get('pass'),
        'all_strands_present': _ok1 and all(
            e.get('curriculum', {}).get(s, 0) > 0 for s in STRAND_NAMES),
        'tooks_ran': _ok1 and e.get('took_competence', {}).get('n', 0) > 0
                     and e.get('took_lexicon', {}).get('n', 0) > 0,
        'parses_ok': _ok1 and sum(
            m['named'] for m in e['post']['arm_meta'].values()) >= 0.5 * sum(
            m['n'] for m in e['post']['arm_meta'].values()),
        'catch_ran': _ok1 and e['post']['catch']['n'] == 12,
        'no_heldout_gen_rows': _ok1 and not any(
            str(r.get('block', '')).startswith('heldout')
            for r in RESULTS['real'].get('injection_rows', [])),
        'shipped': _ok1 and (SEM / INFLIGHT / 'condition_real.json').exists(),
        'adapter_saved': _ok1 and (SEM / INFLIGHT / 'readout_real' /
                                   'adapter_config.json').exists(),
    }
    ok = all(checks.values())
    print('smoke checks:', json.dumps(checks, indent=1))
    banner = ('SMOKE GREEN — flip SMOKE=False, Runtime > Restart runtime, '
              'Run all. Full mode = ONE run (~90-120 min worst case; the '
              'per-strand plateau may stop earlier).'
              if ok else 'SMOKE RED — do not fly full; send Fable the output')
    print('\n' + '=' * 66 + f'\n  {banner}\n' + '=' * 66)
elif COMPLETE == ['real']:
    print('\nFULL FLIGHT COMPLETE — shipped to MyDrive/semcore/e8n3/')
    print('fork:', fork)
else:
    print('\n' + '=' * 66)
    print('  INCOMPLETE — real did not land. Next run: Runtime > Restart '
          'runtime,')
    print(f"  set RESUME_STAMP = '{RESUME_STAMP or STAMP}', then Run all.")
    print('=' * 66)
